# PawSafe — GMM 기반 상대적 Heat Cost 산책로 분석

## 2026-08-15 16:00 KST · 송파구 전체 보행 Edge

이 Notebook은 원본 PawSafe 프로젝트의 **Raw Data 로딩·공간 전처리·그늘·일사·축열·5개 Feature 계산까지만** 같은 로직과 파라미터로 재현하고, 그 이후는 새 분석 파이프라인으로 분리한다.

- 기존과 동일하게 유지: `shade_ratio`, `recent_direct_sun_minutes`, `cumulative_effective_solar_mj_m2`, `heat_storage_proxy`, `surface_absorptivity`
- 새로 수행: Pearson 상관관계 + VIF 기반 변수 선택 → 선택된 2~3개 Feature의 표준화 → GMM 3군집 → 상대 Heat Cost 0/1/2 → 두 Dijkstra 경로 비교
- 사용하지 않음: 기존 PCA Heat Cost, 기존 0~100 산식, PCA 기반 다중공선성 처리, KMeans, 저장 모델·cluster label, fast/balanced/cool 가중치 및 기존 경로 결과

> **해석 한계:** Heat Cost는 실제 노면온도(℃) 예측값이 아니다. 그늘·직사노출·누적일사·축열·포장 흡수 특성으로 나눈 군집 사이의 **상대적인 열노출 순위**다. Silhouette Score도 온도 예측 정확도가 아니라 Feature 공간에서 군집이 얼마나 분리됐는지를 나타내는 내부 평가 지표다.

각 주요 Code Cell 앞의 Markdown은 입력, 처리 목적, 출력, 계산 의미와 해석 주의사항을 설명한다. 위에서 아래로 순서대로 실행한다.


## 0. 실행 환경과 프로젝트 위치 확인

이 셀은 Notebook 실행에 필요한 패키지를 확인하고, 누락된 패키지만 설치한다. 이어서 현재 폴더 또는 Colab의 `/content` 아래에서 `data/raw/songpa_walkways.gpkg`를 찾아 프로젝트 루트로 이동한다. Colab에 프로젝트가 없다면 ZIP 업로드 창을 열고 안전하게 압축을 해제한다.

입력은 이 Notebook이 포함된 프로젝트 폴더 또는 프로젝트 ZIP이며, 출력은 이후 모든 상대경로의 기준이 되는 `PROJECT_ROOT`다. ZIP 안에는 원본 Raw Data가 포함되어 있으므로 별도의 파일명 변경이 필요 없다.


In [ ]:
import importlib.util
import os
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REQUIRED_MODULES = {
    "geopandas": "geopandas",
    "pyogrio": "pyogrio",
    "pyarrow": "pyarrow",
    "pysolar": "pysolar",
    "networkx": "networkx",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
    "seaborn": "seaborn",
    "requests": "requests",
}
missing_packages = [package for module, package in REQUIRED_MODULES.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    print("누락 패키지 설치:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])


def locate_project():
    candidates = [Path.cwd(), Path("/content"), Path("/content/pawsafe_gmm_heat_route")]
    for base in candidates:
        if (base / "data" / "raw" / "songpa_walkways.gpkg").exists():
            return base.resolve()
    if Path("/content").exists():
        for walkway in Path("/content").glob("**/data/raw/songpa_walkways.gpkg"):
            return walkway.parents[2].resolve()
    return None


def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"안전하지 않은 ZIP 경로: {member.filename}")
        archive.extractall(destination)


PROJECT_ROOT = locate_project()
if PROJECT_ROOT is None and IN_COLAB:
    from google.colab import files
    print("PawSafe 프로젝트 ZIP을 업로드하세요.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("업로드된 ZIP이 없습니다.")
    for zip_name in zip_names:
        safe_extract_zip(zip_name, "/content")
    PROJECT_ROOT = locate_project()

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "data/raw/songpa_walkways.gpkg를 찾지 못했습니다. 프로젝트 폴더에서 실행하거나 ZIP을 업로드하세요."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("프로젝트 루트:", PROJECT_ROOT)


## 0-1. 공통 import, 한글 폰트, 분석 CONFIG

이 셀은 전체 Notebook에서 공통으로 쓰는 라이브러리와 설정값을 한곳에 모은다. `TARGET_TIME`, 출발·도착 좌표, `HEAT_PENALTY`는 사용자가 가장 자주 바꿀 값이다. 좌표는 위도(`LAT`)와 경도(`LON`) 순서에 유의한다.

그늘·축열 관련 값은 원본 프로젝트와 동일하다: EPSG:5186, 20m Edge 샘플링, 건물/가로수 최대 그림자 180m/30m, 가로수 투과율 0.30, 최근 직사광 3시간, 누적일사 6시간, thermal memory 2.5시간, 풍속·강수 냉각계수 0.12/0.35. 이 값들은 임의의 새 공식으로 바꾸지 않는다.

`CORR_THRESHOLD=0.85`, `VIF_THRESHOLD=10`은 새 변수 선택 단계의 기준이고, `HEAT_PENALTY`는 Heat Cost 경로의 거리 대비 열노출 회피 강도를 조절한다. `ROUTE_ALERT_*` 설정은 다익스트라가 경로를 확정한 **후** 사용자 화면에 보낼 1~100점과 40/80 경계만 제어하며, 그래프 가중치나 경로 선택에는 관여하지 않는다. 모든 결과 폴더는 없으면 자동 생성된다.


In [ ]:
import hashlib
import json
import math
import random
import time
import warnings
from datetime import timedelta, timezone
from getpass import getpass
from urllib.parse import unquote

import geopandas as gpd
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from IPython.display import Markdown, display
from matplotlib.colors import ListedColormap
from pysolar.solar import get_altitude, get_azimuth
from shapely.geometry import LineString, Point
from shapely.ops import unary_union
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from pawsafe_loaders import (
    attach_pavement,
    load_boundary,
    load_buildings,
    load_trees,
    load_walkways,
    normalize_weather,
    read_csv_auto,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def setup_korean_font():
    preferred = ["NanumGothic", "Noto Sans CJK KR", "Malgun Gothic", "AppleGothic"]
    available = {font.name for font in fm.fontManager.ttflist}
    for name in preferred:
        if name in available:
            plt.rcParams["font.family"] = name
            return name
    if IN_COLAB:
        subprocess.run(["apt-get", "install", "-qq", "-y", "fonts-nanum"], check=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
        plt.rcParams["font.family"] = "NanumGothic"
        return "NanumGothic"
    return "기본 폰트(한글 폰트 설치 권장)"


FONT_NAME = setup_korean_font()
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font=plt.rcParams["font.family"][0]
              if isinstance(plt.rcParams["font.family"], list)
              else plt.rcParams["font.family"])

# ── 사용자가 주로 변경하는 값 ───────────────────────────────────
TARGET_TIME = "2026-08-15 16:00:00"
START_LAT = 37.49278694313202
START_LON = 127.15255448773718
END_LAT = 37.47883618601743
END_LON = 127.12871618594451
HEAT_PENALTY = 1.0
CORR_THRESHOLD = 0.85
VIF_THRESHOLD = 10.0

# ── 최종 추천 경로의 사용자 경고 후처리 전용 ─────────────────────
# 기존 다익스트라 가중치와 경로에는 사용하지 않는다.
ROUTE_ALERT_ALPHA = 0.5
ROUTE_ALERT_TEMP_MIN_C = 0.0
ROUTE_ALERT_TEMP_MAX_C = 50.0
ROUTE_COMFORT_MAX_SCORE = 40
ROUTE_WARNING_MIN_SCORE = 80

# ── 원본 프로젝트와 동일한 전처리·5 Feature 파라미터 ─────────────
CRS = "EPSG:5186"
WGS84 = "EPSG:4326"
KST = timezone(timedelta(hours=9))
LOCATION_LAT = 37.5145
LOCATION_LON = 127.1060
SHADOW_INTERVAL_MINUTES = 60
RECENT_SUN_WINDOW_HOURS = 3
CUMULATIVE_WINDOW_HOURS = 6
INFERENCE_HISTORY_HOURS = 12
EDGE_SAMPLE_SPACING_M = 20
MAX_BUILDING_SHADOW_M = 180
MAX_TREE_SHADOW_M = 30
TREE_TRANSMISSIVITY = 0.30
THERMAL_MEMORY_HOURS = 2.5
WIND_COOLING_FACTOR = 0.12
RAIN_COOLING_FACTOR = 0.35
NODE_SNAP_M = 0.5
ROUTE_INPUT_MAX_SNAP_M = 200.0
ASOS_STATION_ID = "108"
ASOS_ENDPOINT = "https://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"

SURFACE_ABSORPTIVITY = {
    "SWB001": 0.70, "SWB002": 0.68, "SWB003": 0.65,
    "SWB004": 0.72, "SWB005": 0.88, "SWB006": 0.82,
    "SWB007": 0.66, "SWB008": 0.75, "unknown": 0.75,
}
GEO_CONFIG = {
    "edge_sample_spacing_m": EDGE_SAMPLE_SPACING_M,
    "max_building_shadow_m": MAX_BUILDING_SHADOW_M,
    "max_tree_shadow_m": MAX_TREE_SHADOW_M,
    "default_floor_height_m": 3.0,
    "default_building_height_m": 9.0,
    "default_tree_height_m": 8.0,
    "default_tree_crown_width_m": 4.0,
    "tree_transmissivity": TREE_TRANSMISSIVITY,
    "pavement_coordinate_scale": 100.0,
    "pavement_join_max_distance_m": 50.0,
    "node_snap_m": NODE_SNAP_M,
    "route_input_max_snap_m": ROUTE_INPUT_MAX_SNAP_M,
}

FEATURES = [
    "shade_ratio",
    "recent_direct_sun_minutes",
    "cumulative_effective_solar_mj_m2",
    "heat_storage_proxy",
    "surface_absorptivity",
]
FEATURE_LABELS = {
    "shade_ratio": "그늘 비율",
    "recent_direct_sun_minutes": "최근 직사광 노출(분)",
    "cumulative_effective_solar_mj_m2": "누적 유효일사(MJ/m²)",
    "heat_storage_proxy": "축열 지표",
    "surface_absorptivity": "포장 흡수율",
}
THERMAL_DIRECTION = {
    "shade_ratio": -1,
    "recent_direct_sun_minutes": 1,
    "cumulative_effective_solar_mj_m2": 1,
    "heat_storage_proxy": 1,
    "surface_absorptivity": 1,
}

RAW_DIR = Path("data/raw")
OUTPUT_DIR = Path("outputs")
PROCESSED_DIR = OUTPUT_DIR / "processed"
TABLE_DIR = OUTPUT_DIR / "tables"
PPT_DIR = OUTPUT_DIR / "ppt_figures"
for folder in [OUTPUT_DIR, PROCESSED_DIR, TABLE_DIR, PPT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET_TS = pd.Timestamp(TARGET_TIME)
TARGET_TAG = TARGET_TS.strftime("%m%d_%H%M")
SHADOW_CACHE = PROCESSED_DIR / f"shadow_cache_{TARGET_TAG}.parquet"


def save_figure(fig, filename):
    path = PPT_DIR / filename
    fig.savefig(path, dpi=240, bbox_inches="tight", facecolor="white")
    print("그림 저장:", path)
    return path


print("한글 폰트:", FONT_NAME)
print("대상 시각:", TARGET_TS, "KST")
print("출력 폴더:", OUTPUT_DIR.resolve())


## 1. Raw Data 파일 존재 여부와 기본 스키마 검증

이 셀은 경계, 보행망, 건물, 건축물대장, 가로수, 포장재, 포장재 코드, 동봉 ASOS 자료가 실제 경로에 모두 존재하는지 먼저 확인한다. 파일이 하나라도 빠지면 후속 계산을 진행하지 않고 누락 파일명을 명확히 보고한다.

CSV 입력은 원본 로더의 자동 인코딩 판별을 사용한다. 이 단계의 출력은 Raw 파일별 크기, 행 수, 주요 컬럼과 결측 셀 수다. 동봉 ASOS CSV는 API 대체값이 아니라 스키마 확인과 원본 프로젝트 재현성 점검용이며, 최종 2026-08-15 16:00 분석에는 뒤에서 Open API로 받은 연속 이력을 사용한다.


In [ ]:
FILES = {
    "boundary": RAW_DIR / "songpa_boundary.gpkg",
    "walkways": RAW_DIR / "songpa_walkways.gpkg",
    "buildings": RAW_DIR / "buildings_songpa.parquet",
    "building_register": RAW_DIR / "building_register_songpa.csv",
    "street_trees": RAW_DIR / "street_trees.csv",
    "pavement": RAW_DIR / "SWM_WKAR_AS.csv",
    "pavement_codes": RAW_DIR / "SWM_BASIC_CODE.csv",
    "asos_backup": RAW_DIR / "asos_hourly.csv",
}
missing_files = [str(path) for path in FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError("필수 Raw Data 누락:\n- " + "\n- ".join(missing_files))

pavement_raw = read_csv_auto(FILES["pavement"])
pavement_codes_raw = read_csv_auto(FILES["pavement_codes"])
register_raw = read_csv_auto(FILES["building_register"])
trees_raw_preview = read_csv_auto(FILES["street_trees"])
asos_backup_raw = read_csv_auto(FILES["asos_backup"])

raw_inventory = pd.DataFrame([
    {
        "dataset": name,
        "path": str(path),
        "size_mb": path.stat().st_size / 1024**2,
    }
    for name, path in FILES.items()
])
display(raw_inventory.round({"size_mb": 3}))

csv_schema = pd.DataFrame([
    {"dataset": "building_register", "rows": len(register_raw),
     "missing_cells": int(register_raw.isna().sum().sum()),
     "major_columns": ", ".join(map(str, register_raw.columns[:12]))},
    {"dataset": "street_trees", "rows": len(trees_raw_preview),
     "missing_cells": int(trees_raw_preview.isna().sum().sum()),
     "major_columns": ", ".join(map(str, trees_raw_preview.columns[:12]))},
    {"dataset": "pavement", "rows": len(pavement_raw),
     "missing_cells": int(pavement_raw.isna().sum().sum()),
     "major_columns": ", ".join(map(str, pavement_raw.columns[:12]))},
    {"dataset": "pavement_codes", "rows": len(pavement_codes_raw),
     "missing_cells": int(pavement_codes_raw.isna().sum().sum()),
     "major_columns": ", ".join(map(str, pavement_codes_raw.columns[:12]))},
    {"dataset": "asos_backup", "rows": len(asos_backup_raw),
     "missing_cells": int(asos_backup_raw.isna().sum().sum()),
     "major_columns": ", ".join(map(str, asos_backup_raw.columns[:12]))},
])
display(csv_schema)


## 1-1. 원본과 동일한 정적 공간 전처리

이 셀은 동봉된 `pawsafe_loaders.py`를 그대로 호출한다. 송파구 경계를 EPSG:5186으로 통일한 뒤 보행 가능 유형만 남기고, LineString을 `edge_id`와 실제 미터 길이 `length_m`로 구성한다. 건물 높이는 실측 → 건축물대장 보완 → 층수×3m → 기본 9m 순서로 채우며, 가로수 수고·수관폭도 원본의 clip 및 기본값을 사용한다.

포장재는 원본과 같이 EPSG:5181 좌표를 100으로 나눈 후 Edge와 가장 긴 교차 구간을 우선 매칭하고, 남은 Edge는 50m 이내 최근접 포장재를 사용한다. 코드별 `surface_absorptivity`도 원본 값 그대로다. 출력 표에서 행 수, CRS, 결측, 주요 컬럼, Edge/건물/가로수 개수와 포장재 매칭률을 확인한다.


In [ ]:
boundary_source = gpd.read_file(FILES["boundary"]).to_crs(CRS)
boundary = load_boundary(FILES["boundary"], CRS)
edges = load_walkways(FILES["walkways"], boundary, CRS)
buildings = load_buildings(
    FILES["buildings"], boundary, CRS, GEO_CONFIG,
    register_path=FILES["building_register"],
)
trees = load_trees(FILES["street_trees"], boundary, CRS, GEO_CONFIG)
edges = attach_pavement(
    edges, FILES["pavement"], FILES["pavement_codes"],
    CRS, GEO_CONFIG, SURFACE_ABSORPTIVITY,
)
weather_backup = normalize_weather(asos_backup_raw)

if edges["edge_id"].duplicated().any():
    duplicated = edges.loc[edges["edge_id"].duplicated(), "edge_id"].tolist()[:10]
    raise ValueError(f"edge_id 중복 발견: {duplicated}")
if (edges["length_m"] <= 0).any():
    raise ValueError("length_m가 0 이하인 Edge가 있습니다.")

known_surface = edges["surface_code"].ne("unknown")
surface_match_edge_pct = 100 * known_surface.mean()
surface_match_length_pct = (
    100 * edges.loc[known_surface, "length_m"].sum() / edges["length_m"].sum()
)
edge_fingerprint = hashlib.sha256(
    "|".join(f"{eid}:{length:.3f}" for eid, length in
             zip(edges["edge_id"], edges["length_m"])).encode("utf-8")
).hexdigest()

data_quality = pd.DataFrame([
    {"dataset": "송파구 경계", "rows": len(boundary_source), "crs": str(boundary_source.crs),
     "missing_cells": int(boundary_source.isna().sum().sum()),
     "major_columns": ", ".join(map(str, boundary_source.columns[:12]))},
    {"dataset": "보행 Edge", "rows": len(edges), "crs": str(edges.crs),
     "missing_cells": int(edges.isna().sum().sum()),
     "major_columns": ", ".join(map(str, edges.columns[:12]))},
    {"dataset": "건물", "rows": len(buildings), "crs": str(buildings.crs),
     "missing_cells": int(buildings.isna().sum().sum()),
     "major_columns": ", ".join(map(str, buildings.columns[:12]))},
    {"dataset": "가로수", "rows": len(trees), "crs": str(trees.crs),
     "missing_cells": int(trees.isna().sum().sum()),
     "major_columns": ", ".join(map(str, trees.columns[:12]))},
    {"dataset": "건축물대장", "rows": len(register_raw), "crs": "비공간 CSV",
     "missing_cells": int(register_raw.isna().sum().sum()),
     "major_columns": ", ".join(map(str, register_raw.columns[:12]))},
    {"dataset": "포장재", "rows": len(pavement_raw), "crs": "원자료 EPSG:5181",
     "missing_cells": int(pavement_raw.isna().sum().sum()),
     "major_columns": ", ".join(map(str, pavement_raw.columns[:12]))},
    {"dataset": "포장재 코드", "rows": len(pavement_codes_raw), "crs": "비공간 CSV",
     "missing_cells": int(pavement_codes_raw.isna().sum().sum()),
     "major_columns": ", ".join(map(str, pavement_codes_raw.columns[:12]))},
    {"dataset": "동봉 ASOS", "rows": len(weather_backup), "crs": "비공간 시계열",
     "missing_cells": int(weather_backup.isna().sum().sum()),
     "major_columns": ", ".join(weather_backup.columns)},
])
display(data_quality)
display(edges[["edge_id", "length_m", "surface_code", "surface_absorptivity"]].head())
print(f"Edge {len(edges):,}개 · 총 길이 {edges.length_m.sum()/1000:.1f} km")
print(f"건물 {len(buildings):,}동 · 가로수 {len(trees):,}그루")
print(f"포장재 매칭: Edge {surface_match_edge_pct:.1f}% · 거리 {surface_match_length_pct:.1f}%")
print("Edge fingerprint:", edge_fingerprint[:16])


## 1-2. 보행 그래프 생성과 연결성 구조 확인

경로 탐색을 위해 각 LineString의 연속 좌표쌍을 그래프 구간으로 만든다. 원본과 동일하게 0.5m 격자로 좌표를 snap하고, 강제로 모든 기하 교차점을 noding하지 않는다. 각 그래프 구간은 부모 `edge_id`와 실제 구간 길이를 보존한다.

출력은 Node 수, 그래프 구간 수, 연결요소 수와 최대 연결요소 크기다. 뒤에서 출발·도착점을 가장 가까운 Node에 붙인 뒤 같은 연결요소인지 다시 검사한다. 서로 다른 연결요소라면 임의 연결하지 않고 오류를 낸다.


In [ ]:
def build_graph(edge_gdf):
    snap = NODE_SNAP_M
    key = lambda point: (
        round(point[0] / snap) * snap,
        round(point[1] / snap) * snap,
    )
    graph = nx.MultiGraph()
    for row in edge_gdf.itertuples():
        coords = list(row.geometry.coords)
        for p, q in zip(coords[:-1], coords[1:]):
            a, b = key(p), key(q)
            if a == b:
                continue
            graph.add_edge(
                a,
                b,
                edge_id=row.edge_id,
                length_m=float(np.hypot(q[0] - p[0], q[1] - p[1])),
            )
    return graph


G = build_graph(edges)
NODE_ARRAY = np.asarray(list(G.nodes), dtype=float)
components = sorted(nx.connected_components(G), key=len, reverse=True)
component_rank = {
    node: rank
    for rank, nodes in enumerate(components, start=1)
    for node in nodes
}


def nearest_node(lon, lat, max_distance_m=ROUTE_INPUT_MAX_SNAP_M):
    point = gpd.GeoSeries([Point(float(lon), float(lat))], crs=WGS84).to_crs(CRS).iloc[0]
    distances = np.hypot(NODE_ARRAY[:, 0] - point.x, NODE_ARRAY[:, 1] - point.y)
    index = int(distances.argmin())
    if distances[index] > max_distance_m:
        raise ValueError(
            f"입력 좌표에서 가장 가까운 보행 Node까지 {distances[index]:.1f}m로, "
            f"허용값 {max_distance_m:.1f}m를 초과합니다."
        )
    return tuple(NODE_ARRAY[index]), float(distances[index])


print(f"Node {G.number_of_nodes():,} · 그래프 구간 {G.number_of_edges():,}")
print(f"연결요소 {len(components):,}개 · 최대 연결요소 {len(components[0]):,} Node")
display(pd.DataFrame({
    "component_rank": range(1, min(11, len(components) + 1)),
    "node_count": [len(c) for c in components[:10]],
}))


## 2. ASOS Open API 함수와 컬럼 정규화

이 셀은 기상청 ASOS 시간자료 API의 서울 관측소(`stnIds=108`)를 호출하는 함수를 정의한다. 요청 범위는 최종 대상 시각만이 아니라 `TARGET_TIME - 12시간`부터 대상 시각까지의 **13개 연속 시각**이다. 최근 직사광, 누적 유효일사, thermal memory를 제대로 warm-up하려면 이전 이력이 반드시 필요하다.

API 원자료의 `ta`, `hm`, `ws`, `rn`, `icsr`, `ss`를 원본 로더와 같은 표준 컬럼으로 바꾼다. 전체 시간행이 하나라도 누락되면 누락 시각을 출력하고 즉시 중단한다. 행 자체가 없는 시간을 임의 생성하거나 잘못된 값으로 채우지 않는다. 다만 존재하는 관측행 안의 결측 처리(강수·일사 0, 기온·습도·풍속 보간)는 원본 `normalize_weather()`의 규칙을 그대로 따른다.


In [ ]:
ASOS_COLUMNS = {
    "ta": "air_temperature_c",
    "hm": "humidity_pct",
    "ws": "wind_speed_ms",
    "rn": "rainfall_mm",
    "icsr": "solar_radiation_mj_m2",
    "ss": "sunshine_hours",
}


def fetch_asos_range(start_time, end_time, service_key, timeout=30):
    response = requests.get(
        ASOS_ENDPOINT,
        timeout=timeout,
        params={
            "serviceKey": service_key,
            "pageNo": 1,
            "numOfRows": 300,
            "dataType": "JSON",
            "dataCd": "ASOS",
            "dateCd": "HR",
            "startDt": pd.Timestamp(start_time).strftime("%Y%m%d"),
            "startHh": pd.Timestamp(start_time).strftime("%H"),
            "endDt": pd.Timestamp(end_time).strftime("%Y%m%d"),
            "endHh": pd.Timestamp(end_time).strftime("%H"),
            "stnIds": ASOS_STATION_ID,
        },
    )
    request_url_upper = (response.request.url or "").upper()
    if any(token in request_url_upper for token in ("%252B", "%252F", "%253D")):
        raise RuntimeError("ASOS 서비스 키가 이중 URL 인코딩되었습니다.")
    response.raise_for_status()
    try:
        payload = response.json()["response"]
    except (ValueError, KeyError, TypeError) as exc:
        raise RuntimeError("ASOS JSON 응답을 해석하지 못했습니다.") from exc
    result_code = payload.get("header", {}).get("resultCode")
    if result_code not in (None, "00"):
        message = payload.get("header", {}).get("resultMsg")
        raise RuntimeError(f"ASOS API 오류 {result_code}: {message}")
    items = payload.get("body", {}).get("items", {}).get("item", [])
    if isinstance(items, dict):
        items = [items]
    if not items:
        raise RuntimeError("ASOS 조회 결과가 비어 있습니다.")

    raw = pd.DataFrame(items)
    out = pd.DataFrame({"timestamp": pd.to_datetime(raw["tm"])})
    for source, target in ASOS_COLUMNS.items():
        out[target] = pd.to_numeric(raw[source], errors="coerce") if source in raw else np.nan
    return normalize_weather(out).drop_duplicates("timestamp").reset_index(drop=True)


def get_target_weather(target_time, service_key):
    target = pd.Timestamp(target_time)
    start = target - pd.Timedelta(hours=INFERENCE_HISTORY_HOURS)
    received = fetch_asos_range(start, target, service_key)
    expected = pd.date_range(start, target, freq="h")
    actual = pd.DatetimeIndex(received["timestamp"])
    missing = expected.difference(actual)
    extras = actual.difference(expected)
    if len(missing):
        formatted = [ts.strftime("%Y-%m-%d %H:%M") for ts in missing]
        print("누락 ASOS 시각:", formatted)
        raise RuntimeError("축열 warm-up에 필요한 연속 ASOS 시각이 누락됐습니다.")
    if len(extras):
        received = received[received["timestamp"].isin(expected)].copy()
    window = received.sort_values("timestamp").reset_index(drop=True)
    if len(window) != INFERENCE_HISTORY_HOURS + 1:
        raise RuntimeError(f"ASOS 이력 행 수 불일치: {len(window)}")
    required_values = ["air_temperature_c", "humidity_pct", "wind_speed_ms",
                       "rainfall_mm", "solar_radiation_mj_m2"]
    if window[required_values].isna().any().any():
        bad = window.loc[window[required_values].isna().any(axis=1),
                         ["timestamp", *required_values]]
        display(bad)
        raise RuntimeError("정규화 후에도 필수 기상값이 결측입니다.")
    return window, target


## 2-1. 2026-08-15 16:00과 이전 12시간 ASOS 이력 요청

이 셀은 실제 API 요청을 실행한다. 서비스키는 `KMA_ASOS_SERVICE_KEY` 환경변수에서 먼저 찾고, 없으면 `getpass()`로 화면에 노출되지 않게 입력받는다. Encoding 키와 Decoding 키 모두 받을 수 있도록 한 번만 `unquote()`하며 키 자체는 출력하거나 파일에 저장하지 않는다.

출력은 2026-08-15 04:00~16:00 KST의 13개 연속 관측행과 시각 범위다. 최종 GMM과 경로 추천은 이 이력으로 만든 **16:00 Edge snapshot**만 사용한다.


In [ ]:
raw_service_key = os.environ.get("KMA_ASOS_SERVICE_KEY", "").strip()
if not raw_service_key:
    raw_service_key = getpass("ASOS 서비스 키(화면에 표시되지 않음): ").strip()
if not raw_service_key:
    raise ValueError("ASOS 서비스 키가 필요합니다.")
ASOS_SERVICE_KEY = unquote(raw_service_key)

weather_window, actual_target = get_target_weather(TARGET_TS, ASOS_SERVICE_KEY)
print(
    f"ASOS 연속 이력 {len(weather_window)}개: "
    f"{weather_window.timestamp.min()} ~ {weather_window.timestamp.max()} KST"
)
display(weather_window)


## 2-2. 분석에 사용된 ASOS 이력 시각화

이 그림은 16:00 snapshot이 어떤 12시간 warm-up을 거쳤는지 보여준다. 기온·습도, 시간당 일사, 풍속·강수를 함께 그려 축열 입력과 냉각 조건을 확인한다. 일사는 Edge별 유효일사를 만들기 전의 관측소 값이며, 뒤에서 그늘 비율과 포장 흡수율이 곱해진다.

출력은 화면 그래프와 `outputs/ppt_figures/01_asos_history_0815_1600.png`다. 단위와 대상 시각 수직선을 포함해 PPT에서 바로 사용할 수 있게 저장한다.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13.33, 9), sharex=True)
axes[0].plot(weather_window["timestamp"], weather_window["air_temperature_c"],
             color="#D95F02", marker="o", label="기온")
axes[0].set_ylabel("기온 (℃)")
ax_h = axes[0].twinx()
ax_h.plot(weather_window["timestamp"], weather_window["humidity_pct"],
          color="#1F78B4", marker="s", alpha=0.75, label="습도")
ax_h.set_ylabel("습도 (%)")
axes[0].set_title("ASOS 기온·습도")

axes[1].bar(weather_window["timestamp"], weather_window["solar_radiation_mj_m2"],
            width=0.03, color="#F2B134", label="일사")
axes[1].set_ylabel("일사 (MJ/m²)")
axes[1].set_title("시간당 일사 — Edge별 유효일사의 원 입력")

axes[2].plot(weather_window["timestamp"], weather_window["wind_speed_ms"],
             color="#2A9D8F", marker="o", label="풍속")
axes[2].set_ylabel("풍속 (m/s)")
ax_r = axes[2].twinx()
ax_r.bar(weather_window["timestamp"], weather_window["rainfall_mm"],
         width=0.02, color="#457B9D", alpha=0.35, label="강수")
ax_r.set_ylabel("강수 (mm)")
axes[2].set_title("풍속·강수 — 원본 축열 감쇠의 냉각 입력")
axes[2].set_xlabel("KST 관측시각")

for axis in axes:
    axis.axvline(actual_target, color="#C1121F", linestyle="--", linewidth=1.5)
    axis.grid(alpha=0.25)
fig.suptitle(f"PawSafe 분석 ASOS 이력 · 대상 {actual_target} KST", fontsize=16, fontweight="bold")
plt.tight_layout()
save_figure(fig, "01_asos_history_0815_1600.png")
plt.show()
plt.close(fig)


## 3. 태양 위치, 건물·가로수 그림자, Edge 그늘 비율 함수

이 셀은 원본 프로젝트의 그늘 계산을 그대로 재현한다. 송파구 중심 좌표에서 KST 태양 고도·방위각을 구하고, 건물 Polygon 또는 가로수 수관 원을 태양 반대 방향으로 이동시킨 뒤 원형과 이동형의 convex hull을 그림자로 사용한다. 고도가 매우 낮을 때 그림자가 무한히 길어지는 것을 막는 `tan` 하한 0.08과 건물/수목 최대길이 180m/30m도 원본과 같다.

각 Edge는 20m 간격에 해당하도록 최소 2개 점으로 균등 sampling한다. 건물 그림자 점은 1.0, 가로수 그림자 점은 `1 - 투과율(0.30) = 0.70`의 그늘 기여를 갖는다. 건물 그림자를 먼저 적용해 중복 계산하지 않는다. 야간에는 직사광이 없으므로 원본과 같이 `shade_ratio=1`이다.


In [ ]:
def solar_position(timestamp):
    aware = pd.Timestamp(timestamp).to_pydatetime().replace(tzinfo=KST)
    return (
        get_altitude(LOCATION_LAT, LOCATION_LON, aware),
        get_azimuth(LOCATION_LAT, LOCATION_LON, aware),
    )


def _shadow_offset(height, altitude, azimuth, cap):
    tangent = max(math.tan(math.radians(altitude)), 0.08)
    length = min(cap, height / tangent)
    radians = math.radians(azimuth)
    return -length * math.sin(radians), -length * math.cos(radians)


def _sweep(geometry, dx, dy):
    moved = gpd.GeoSeries([geometry], crs=CRS).translate(dx, dy).iloc[0]
    return unary_union([geometry, moved]).convex_hull


def shadow_union(building_gdf, tree_gdf, altitude, azimuth):
    building_shadows = [
        _sweep(geometry, *_shadow_offset(height, altitude, azimuth, MAX_BUILDING_SHADOW_M))
        for geometry, height in zip(building_gdf.geometry, building_gdf["height_m"])
    ]
    tree_shadows = [
        _sweep(
            geometry.buffer(crown_width / 2),
            *_shadow_offset(height, altitude, azimuth, MAX_TREE_SHADOW_M),
        )
        for geometry, height, crown_width in zip(
            tree_gdf.geometry, tree_gdf["height_m"], tree_gdf["crown_width_m"]
        )
    ]
    return (
        unary_union(building_shadows) if building_shadows else None,
        unary_union(tree_shadows) if tree_shadows else None,
    )


def shade_ratio_by_edge(edge_gdf, building_shadow, tree_shadow):
    from shapely.prepared import prep
    if building_shadow is None and tree_shadow is None:
        return np.zeros(len(edge_gdf))
    prepared_building = prep(building_shadow) if building_shadow is not None else None
    prepared_tree = prep(tree_shadow) if tree_shadow is not None else None
    ratios = []
    for geometry in edge_gdf.geometry:
        n_points = max(2, int(math.ceil(geometry.length / EDGE_SAMPLE_SPACING_M)) + 1)
        points = [geometry.interpolate(i / (n_points - 1), normalized=True)
                  for i in range(n_points)]
        shaded = 0.0
        for point in points:
            if prepared_building is not None and prepared_building.covers(point):
                shaded += 1.0
            elif prepared_tree is not None and prepared_tree.covers(point):
                shaded += 1.0 - TREE_TRANSMISSIVITY
        ratios.append(shaded / n_points)
    return np.asarray(ratios)


## 3-1. 이전 12시간 전체 Edge 그림자 이력 계산

입력은 13개 ASOS 시각, 전처리된 건물·가로수, 전체 보행 Edge다. 각 시각의 태양 위치로 그림자를 만들고 Edge별 `shade_ratio`를 계산한다. 결과는 `edge_id × timestamp`의 긴 형식 DataFrame이다.

그림자 계산은 공간연산량이 크므로 같은 대상시각의 결과를 `outputs/processed/shadow_cache_0815_1600.parquet`에 저장한다. 재실행 시 캐시에 해당 시각의 모든 Edge가 정확히 있을 때만 재사용하고, 빠진 시각은 다시 계산한다. 진행 로그의 평균 그늘과 태양 고도를 확인할 수 있다.


In [ ]:
def compute_shadows(edge_gdf, weather_df, building_gdf, tree_gdf):
    cache = pd.read_parquet(SHADOW_CACHE) if SHADOW_CACHE.exists() else pd.DataFrame()
    if len(cache):
        cache["timestamp"] = pd.to_datetime(cache["timestamp"])
    edge_ids = set(edge_gdf["edge_id"])
    rows, new_rows = [], []

    for number, timestamp in enumerate(weather_df["timestamp"], start=1):
        hit = (
            cache[(cache["timestamp"] == timestamp) & cache["edge_id"].isin(edge_ids)]
            if len(cache) else pd.DataFrame()
        )
        if len(hit) == len(edge_gdf) and hit["edge_id"].nunique() == len(edge_gdf):
            rows.append(hit)
            print(f"[{number:02d}/{len(weather_df):02d}] {timestamp} · 캐시 재사용")
            continue

        started = time.time()
        altitude, azimuth = solar_position(timestamp)
        if altitude <= 0:
            ratios = np.ones(len(edge_gdf))
        else:
            building_shadow, tree_shadow = shadow_union(
                building_gdf, tree_gdf, altitude, azimuth
            )
            ratios = shade_ratio_by_edge(edge_gdf, building_shadow, tree_shadow)
        frame = pd.DataFrame({
            "edge_id": edge_gdf["edge_id"].to_numpy(),
            "timestamp": timestamp,
            "solar_altitude_deg": altitude,
            "solar_azimuth_deg": azimuth,
            "shade_ratio": ratios,
        })
        rows.append(frame)
        new_rows.append(frame)
        print(
            f"[{number:02d}/{len(weather_df):02d}] {timestamp} · "
            f"고도 {altitude:5.1f}° · 평균그늘 {ratios.mean():.3f} · "
            f"{time.time() - started:.0f}s"
        )

    if new_rows:
        added = pd.concat(new_rows, ignore_index=True)
        merged = pd.concat([cache, added], ignore_index=True) if len(cache) else added
        merged = merged.drop_duplicates(["edge_id", "timestamp"], keep="last")
        merged.to_parquet(SHADOW_CACHE, index=False)
        print("그림자 캐시 저장:", SHADOW_CACHE)
    result = pd.concat(rows, ignore_index=True)
    expected_rows = len(edge_gdf) * len(weather_df)
    if len(result) != expected_rows:
        raise ValueError(f"그림자 행 수 불일치: {len(result):,} != {expected_rows:,}")
    return result


shadow_history = compute_shadows(edges, weather_window, buildings, trees)
print("그림자 이력 shape:", shadow_history.shape)
display(shadow_history.head())


## 3-2. 원본 계산식으로 5개 Feature 생성과 16:00 snapshot 추출

이 셀은 원본 프로젝트의 5개 Feature 계산식을 그대로 적용한다.

- `sun_fraction = 1 - shade_ratio`
- `effective_solar = ASOS 일사 × sun_fraction × surface_absorptivity`
- 최근 직사광 노출은 최근 3개 시간의 `sun_fraction` 합 × 60분
- 누적 유효일사는 최근 6개 시간의 `effective_solar` 합
- 축열은 이전 축열을 thermal memory 2.5시간으로 지수 감쇠하고, 풍속(0.12)·강수(0.35)가 감쇠를 더 빠르게 한 뒤 현재 유효일사를 더하는 재귀식

즉, 각 Edge에서 `storage_t = storage_(t-1) × decay_t + effective_solar_t`이고 `decay_t = exp(-(1h/2.5h))^(1 + 0.12×풍속 + 0.35×강수)`다. 13개 이력 전체를 계산하되, 이후 분석에는 정확히 16:00 한 시각의 Edge별 행만 남긴다. NaN/inf, 중복 `edge_id`, 행 수를 검증한 뒤 진행한다.


In [ ]:
def build_features(edge_gdf, shadows_df, weather_df):
    static = edge_gdf[["edge_id", "length_m", "surface_code", "surface_absorptivity"]]
    frame = (
        shadows_df
        .merge(weather_df, on="timestamp", how="left", validate="many_to_one")
        .merge(static, on="edge_id", how="left", validate="many_to_one")
        .sort_values(["edge_id", "timestamp"])
        .reset_index(drop=True)
    )
    frame["sun_fraction"] = (1 - frame["shade_ratio"]).clip(0, 1)
    frame["effective_solar_mj_m2"] = (
        frame["solar_radiation_mj_m2"]
        * frame["sun_fraction"]
        * frame["surface_absorptivity"]
    )

    recent_n = max(1, int(RECENT_SUN_WINDOW_HOURS * 60 / SHADOW_INTERVAL_MINUTES))
    cumulative_n = max(1, int(CUMULATIVE_WINDOW_HOURS * 60 / SHADOW_INTERVAL_MINUTES))
    grouped = frame.groupby("edge_id", group_keys=False)
    frame["recent_direct_sun_minutes"] = (
        grouped["sun_fraction"].rolling(recent_n, min_periods=1).sum()
        .reset_index(level=0, drop=True) * SHADOW_INTERVAL_MINUTES
    )
    frame["cumulative_effective_solar_mj_m2"] = (
        grouped["effective_solar_mj_m2"].rolling(cumulative_n, min_periods=1).sum()
        .reset_index(level=0, drop=True)
    )

    base_decay = math.exp(
        -(SHADOW_INTERVAL_MINUTES / 60) / THERMAL_MEMORY_HOURS
    )
    storage = np.zeros(len(frame))
    for _, indices in frame.groupby("edge_id", sort=False).groups.items():
        accumulator = 0.0
        for index in indices:
            wind = frame.at[index, "wind_speed_ms"]
            rain = frame.at[index, "rainfall_mm"]
            decay = base_decay ** (
                1 + WIND_COOLING_FACTOR * wind + RAIN_COOLING_FACTOR * rain
            )
            accumulator = accumulator * decay + frame.at[index, "effective_solar_mj_m2"]
            storage[index] = accumulator
    frame["heat_storage_proxy"] = storage
    return frame


feature_history = build_features(edges, shadow_history, weather_window)
expected_history_rows = len(edges) * len(weather_window)
if len(feature_history) != expected_history_rows:
    raise ValueError(
        f"Feature 이력 행 수 불일치: {len(feature_history):,} != {expected_history_rows:,}"
    )

target_features = feature_history[feature_history["timestamp"] == actual_target].copy()
if len(target_features) != len(edges):
    raise ValueError(f"대상시각 Edge 수 불일치: {len(target_features):,} != {len(edges):,}")
if target_features["edge_id"].duplicated().any():
    raise ValueError("대상시각 snapshot에 중복 edge_id가 있습니다.")

# surface_absorptivity는 정적 Edge 테이블에 이미 있으므로 병합 이름 충돌을
# 피하고, 나머지 네 시계열 Feature만 target_features에서 가져온다.
dynamic_features = [feature for feature in FEATURES if feature != "surface_absorptivity"]
snapshot_values = target_features[
    ["edge_id", "timestamp", "solar_altitude_deg", "solar_azimuth_deg",
     "air_temperature_c", "humidity_pct", "wind_speed_ms", "rainfall_mm",
     "solar_radiation_mj_m2", *dynamic_features]
]
edge_snapshot = edges.merge(snapshot_values, on="edge_id", how="left", validate="one_to_one")
edge_snapshot = gpd.GeoDataFrame(edge_snapshot, geometry="geometry", crs=CRS)

numeric_features = edge_snapshot[FEATURES].replace([np.inf, -np.inf], np.nan)
if numeric_features.isna().any().any():
    display(edge_snapshot.loc[numeric_features.isna().any(axis=1), ["edge_id", *FEATURES]].head(20))
    raise ValueError("대상시각 5개 Feature에 NaN/inf가 있습니다.")

print(f"Feature 이력: {len(feature_history):,}행")
print(f"16:00 snapshot: {len(edge_snapshot):,} Edge")
display(edge_snapshot[["edge_id", "length_m", *FEATURES]].head())


## 4. 5개 Feature의 물리적 의미와 방향성

아래 5개 값은 실제 노면온도(℃)를 직접 예측하는 변수가 아니라 Edge의 상대적 열노출 특성이다.

| Feature | 물리적 의미 | 값이 커질 때의 일반적 방향 |
|---|---|---|
| `shade_ratio` | Edge sample point가 건물 또는 투과율을 반영한 가로수 그림자에 든 비율 | 그늘 증가 → **cooler** |
| `recent_direct_sun_minutes` | 최근 3시간의 부분 직사광 노출을 분 단위로 환산 | 직사광 증가 → **hotter** |
| `cumulative_effective_solar_mj_m2` | 최근 6시간의 일사×비그늘×포장흡수율 누적 | 흡수 태양에너지 증가 → **hotter** |
| `heat_storage_proxy` | thermal memory, 풍속·강수 냉각을 반영한 재귀 축열 지표 | 축적 열에너지 증가 → **hotter** |
| `surface_absorptivity` | 포장 종류별 태양복사 흡수율 | 흡수율 증가 → **hotter** |

다음 셀은 count, mean, std, min, median, max, 결측 비율을 계산해 CSV로 저장한다. 이 표는 GMM 이전에 값 범위와 데이터 품질을 확인하는 기준이다.


In [ ]:
feature_summary = pd.DataFrame({
    "count": edge_snapshot[FEATURES].count(),
    "mean": edge_snapshot[FEATURES].mean(),
    "std": edge_snapshot[FEATURES].std(),
    "min": edge_snapshot[FEATURES].min(),
    "median": edge_snapshot[FEATURES].median(),
    "max": edge_snapshot[FEATURES].max(),
    "missing_ratio": edge_snapshot[FEATURES].isna().mean(),
})
feature_summary.index.name = "feature"
feature_summary.to_csv(TABLE_DIR / "feature_summary.csv", encoding="utf-8-sig")
display(feature_summary.round(4))
print("표 저장:", TABLE_DIR / "feature_summary.csv")


## 4-1. 전체 Edge `shade_ratio` 지도

입력은 16:00 Edge snapshot의 `shade_ratio`다. 밝은 색은 그늘 비율이 낮고, 진한 청록색은 건물·가로수 그림자 기여가 큰 Edge를 뜻한다. 이 지도는 실제 온도 지도가 아니라 같은 시각의 상대적인 그늘 구조를 보여준다.

출력은 `outputs/ppt_figures/02_shade_ratio_map_0815_1600.png`다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
edge_snapshot.plot(
    ax=ax, column="shade_ratio", cmap="YlGnBu", linewidth=1.1,
    legend=True, legend_kwds={"label": "그늘 비율 (0~1)", "shrink": 0.7},
)
ax.set_title(f"송파구 전체 보행 Edge 그늘 비율 · {actual_target} KST", fontsize=15, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
save_figure(fig, "02_shade_ratio_map_0815_1600.png")
plt.show()
plt.close(fig)


## 4-2. 5개 Feature 분포

각 Feature의 히스토그램과 중앙값 선을 함께 표시해 치우침, 다봉성, 값의 집중 구간을 살핀다. GMM은 Gaussian component를 결합해 분포를 설명하므로 단일 정규분포처럼 보이지 않아도 되지만, 극단값이나 거의 상수인 변수가 있는지 먼저 확인해야 한다.

출력은 `outputs/ppt_figures/03_feature_distributions.png`다. 축에는 각 Feature의 실제 단위를 유지한다.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, feature in zip(axes.flat, FEATURES):
    values = edge_snapshot[feature].dropna()
    sns.histplot(values, bins=35, kde=True, ax=axis, color="#2A9D8F")
    axis.axvline(values.median(), color="#C1121F", linestyle="--",
                 label=f"중앙값 {values.median():.3f}")
    axis.set_title(FEATURE_LABELS[feature])
    axis.set_xlabel(feature)
    axis.set_ylabel("Edge 수")
    axis.legend()
axes.flat[-1].axis("off")
fig.suptitle("2026-08-15 16:00 전체 Edge의 5개 열노출 Feature 분포",
             fontsize=16, fontweight="bold")
plt.tight_layout()
save_figure(fig, "03_feature_distributions.png")
plt.show()
plt.close(fig)


## 5. Pearson 상관관계와 고상관 Feature pair

5개 Feature의 Pearson 상관계수를 계산한다. Heatmap 셀 안에 실제 계수를 표기하고, 절대 상관계수가 큰 pair를 별도 표로 정렬한다. `|r| ≥ 0.85`는 변수 제거를 검토하는 기준이지 자동적인 인과관계나 동일성을 뜻하지 않는다.

입력은 16:00 snapshot의 원래 5개 Feature이며, 출력은 상관행렬 CSV, pair 표, 고해상도 Heatmap이다. 이 단계에서는 아직 변수를 삭제하지 않는다.


In [ ]:
feature_correlation = edge_snapshot[FEATURES].corr(method="pearson")
feature_correlation.to_csv(TABLE_DIR / "feature_correlation.csv", encoding="utf-8-sig")

pairs = []
for i, left in enumerate(FEATURES):
    for right in FEATURES[i + 1:]:
        value = float(feature_correlation.loc[left, right])
        pairs.append({
            "feature_1": left,
            "feature_2": right,
            "correlation": value,
            "abs_correlation": abs(value),
            "above_threshold": abs(value) >= CORR_THRESHOLD,
        })
correlation_pairs = pd.DataFrame(pairs).sort_values("abs_correlation", ascending=False)

display(feature_correlation.round(3))
display(correlation_pairs.reset_index(drop=True).round(3))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    feature_correlation,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Pearson 상관계수"},
)
ax.set_xticklabels([FEATURE_LABELS[name] for name in FEATURES], rotation=30, ha="right")
ax.set_yticklabels([FEATURE_LABELS[name] for name in FEATURES], rotation=0)
ax.set_title("5개 Feature Pearson 상관관계 · 셀 값은 실제 상관계수",
             fontsize=15, fontweight="bold")
plt.tight_layout()
save_figure(fig, "04_feature_correlation_heatmap.png")
plt.show()
plt.close(fig)


## 6. Pearson + VIF 기반 다중공선성 제거 — PCA 사용 안 함

Feature의 물리적 의미를 유지하기 위해 PCA로 변수를 합성하지 않고 실제 변수 이름을 선택한다. 먼저 상수항을 포함한 VIF를 계산한다. 이후 다음 데이터 기반 절차를 반복한다.

1. `|r| ≥ 0.85` pair가 있으면 그 pair에서 VIF가 더 높은 변수를 제거 검토한다.
2. 고상관 pair는 없지만 VIF가 10을 넘으면 VIF가 가장 높은 변수를 제거한다.
3. 기준 위반이 없더라도 4개 이상 남으면 평균 절대상관이 가장 큰, 즉 다른 변수와 가장 중복된 변수를 제거한다.
4. 3개 이하가 된 뒤에도 기준 위반이 있으면 2개까지 줄인다. 최소 2개는 보존한다.

처음부터 특정 Feature를 제거하도록 hard coding하지 않는다. 각 제거 시점의 상관계수, VIF, 중복 상대 Feature와 사유를 기록하고 재계산한다. 출력은 제거 전/후 VIF, 최종 `SELECTED_FEATURES`, 선택 근거 표다.


In [ ]:
def clean_matrix(frame, columns):
    matrix = frame[columns].replace([np.inf, -np.inf], np.nan).copy()
    medians = matrix.median()
    if medians.isna().any():
        raise ValueError(f"중앙값을 계산할 수 없는 Feature: {medians[medians.isna()].index.tolist()}")
    return matrix.fillna(medians)


def compute_vif(frame, columns):
    matrix = clean_matrix(frame, columns)
    design = add_constant(matrix, has_constant="add")
    rows = []
    for offset, feature in enumerate(columns, start=1):
        if matrix[feature].nunique() <= 1:
            value = np.inf
        else:
            try:
                value = float(variance_inflation_factor(design.to_numpy(), offset))
            except Exception:
                value = np.inf
        rows.append({"feature": feature, "vif": value})
    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)


vif_before = compute_vif(edge_snapshot, FEATURES)
selected = FEATURES.copy()
selection_steps = []

while len(selected) > 2:
    current_corr = clean_matrix(edge_snapshot, selected).corr().abs()
    current_vif = compute_vif(edge_snapshot, selected).set_index("feature")["vif"]
    pair_rows = []
    for i, left in enumerate(selected):
        for right in selected[i + 1:]:
            pair_rows.append((left, right, float(current_corr.loc[left, right])))
    pair_rows.sort(key=lambda item: item[2], reverse=True)
    top_left, top_right, top_corr = pair_rows[0]
    max_vif = float(current_vif.max())

    threshold_violation = top_corr >= CORR_THRESHOLD or max_vif > VIF_THRESHOLD
    if len(selected) <= 3 and not threshold_violation:
        break

    mean_abs_corr = (current_corr.sum(axis=1) - 1) / (len(selected) - 1)
    if top_corr >= CORR_THRESHOLD:
        candidates = [top_left, top_right]
        remove_feature = sorted(
            candidates,
            key=lambda name: (current_vif[name], mean_abs_corr[name], name),
            reverse=True,
        )[0]
        peer = top_right if remove_feature == top_left else top_left
        reason = (
            f"|r|={top_corr:.3f}인 {peer}와 정보가 중복되고, pair 안에서 "
            f"VIF={current_vif[remove_feature]:.3f}가 더 높아 제거"
        )
        trigger = "high_correlation"
        related_corr = top_corr
    elif max_vif > VIF_THRESHOLD:
        remove_feature = current_vif.idxmax()
        peer = current_corr.loc[remove_feature].drop(remove_feature).idxmax()
        related_corr = float(current_corr.loc[remove_feature, peer])
        reason = (
            f"VIF={current_vif[remove_feature]:.3f}가 기준 {VIF_THRESHOLD:g}을 초과; "
            f"가장 관련된 {peer}와 |r|={related_corr:.3f}"
        )
        trigger = "high_vif"
    else:
        remove_feature = mean_abs_corr.idxmax()
        peer = current_corr.loc[remove_feature].drop(remove_feature).idxmax()
        related_corr = float(current_corr.loc[remove_feature, peer])
        reason = (
            f"3개로 축약하기 위해 평균 |r|={mean_abs_corr[remove_feature]:.3f}로 "
            f"중복성이 가장 큰 변수를 제거; 가장 관련된 변수는 {peer}"
        )
        trigger = "dimension_target"

    selection_steps.append({
        "step": len(selection_steps) + 1,
        "removed_feature": remove_feature,
        "related_feature": peer,
        "abs_correlation": related_corr,
        "vif_at_removal": float(current_vif[remove_feature]),
        "trigger": trigger,
        "reason": reason,
    })
    selected.remove(remove_feature)

SELECTED_FEATURES = selected
vif_after = compute_vif(edge_snapshot, SELECTED_FEATURES)
selection_log = pd.DataFrame(selection_steps)

decision_rows = []
removed_reason = {row["removed_feature"]: row["reason"] for row in selection_steps}
for feature in FEATURES:
    retained = feature in SELECTED_FEATURES
    decision_rows.append({
        "feature": feature,
        "selected": retained,
        "decision": "retained" if retained else "removed",
        "reason": (
            "반복 재계산 후 상대적으로 고유한 정보를 유지해 최종 GMM 입력으로 선택"
            if retained else removed_reason[feature]
        ),
    })
selected_features_table = pd.DataFrame(decision_rows)

vif_before.to_csv(TABLE_DIR / "vif_before.csv", index=False, encoding="utf-8-sig")
vif_after.to_csv(TABLE_DIR / "vif_after.csv", index=False, encoding="utf-8-sig")
selected_features_table.to_csv(
    TABLE_DIR / "selected_features.csv", index=False, encoding="utf-8-sig"
)

print("SELECTED_FEATURES =", SELECTED_FEATURES)
display(vif_before.round(3))
display(selection_log.round(3))
display(vif_after.round(3))

reason_lines = [
    f"- **{row['removed_feature']} 제거:** {row['reason']}"
    for row in selection_steps
]
display(Markdown(
    "### 실제 계산값에 따른 선택 근거\n" + "\n".join(reason_lines)
    + f"\n\n최종 GMM 입력은 **{', '.join(SELECTED_FEATURES)}**이다. "
      "이 과정에는 PCA를 사용하지 않았다."
))


## 6-1. 제거 전·후 VIF 비교

막대는 각 Feature의 VIF이며 점선은 기준 10이다. 무한대 VIF는 그래프 표시 상한으로 잘라 `∞`로 라벨링한다. 제거 후 남은 Feature의 VIF가 낮아졌는지 확인하되, VIF는 표본의 선형 중복성 지표이지 물리 타당성 자체를 보장하지 않는다.

출력은 `outputs/ppt_figures/05_vif_comparison.png`다.


In [ ]:
def plot_vif_panel(axis, table, title):
    finite = table.loc[np.isfinite(table["vif"]), "vif"]
    cap = max(VIF_THRESHOLD * 1.5, float(finite.max()) * 1.15 if len(finite) else VIF_THRESHOLD * 1.5)
    values = table["vif"].where(np.isfinite(table["vif"]), cap)
    bars = axis.barh(
        [FEATURE_LABELS.get(name, name) for name in table["feature"]],
        values,
        color=["#E76F51" if value > VIF_THRESHOLD else "#2A9D8F" for value in table["vif"]],
    )
    axis.axvline(VIF_THRESHOLD, color="#C1121F", linestyle="--", label=f"기준 {VIF_THRESHOLD:g}")
    axis.set_title(title)
    axis.set_xlabel("VIF")
    axis.legend()
    for bar, original in zip(bars, table["vif"]):
        label = "∞" if not np.isfinite(original) else f"{original:.2f}"
        axis.text(bar.get_width(), bar.get_y() + bar.get_height()/2, f" {label}", va="center")


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_vif_panel(axes[0], vif_before.sort_values("vif"), "제거 전 5개 Feature VIF")
plot_vif_panel(axes[1], vif_after.sort_values("vif"), "변수 선택 후 VIF")
fig.suptitle("Pearson + VIF 기반 다중공선성 점검", fontsize=16, fontweight="bold")
plt.tight_layout()
save_figure(fig, "05_vif_comparison.png")
plt.show()
plt.close(fig)


## 6-2. 다중공선성 제거 전·후 Feature 구성

이 도식은 원래 5개 Feature와 실제 계산 결과로 남은 2~3개 Feature를 한눈에 비교한다. 초록은 유지, 회색은 제거를 뜻한다. 변수를 합성한 것이 아니라 해석 가능한 원래 Feature 중 일부를 선택했다는 점이 핵심이다.

출력은 `outputs/ppt_figures/06_feature_selection_before_after.png`다.


In [ ]:
selection_matrix = np.array([
    [1] * len(FEATURES),
    [1 if feature in SELECTED_FEATURES else 0 for feature in FEATURES],
])
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.imshow(selection_matrix, cmap=ListedColormap(["#D9D9D9", "#2A9D8F"]), vmin=0, vmax=1, aspect="auto")
ax.set_yticks([0, 1], ["제거 전", "GMM 입력"])
ax.set_xticks(range(len(FEATURES)), [FEATURE_LABELS[name] for name in FEATURES], rotation=25, ha="right")
for row in range(2):
    for col in range(len(FEATURES)):
        label = "유지" if selection_matrix[row, col] else "제거"
        color = "white" if selection_matrix[row, col] else "#555555"
        ax.text(col, row, label, ha="center", va="center", color=color, fontweight="bold")
ax.set_title("다중공선성 제거 전·후 Feature 구성 — PCA 미사용", fontsize=15, fontweight="bold")
ax.set_xticks(np.arange(-0.5, len(FEATURES), 1), minor=True)
ax.set_yticks(np.arange(-0.5, 2, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)
plt.tight_layout()
save_figure(fig, "06_feature_selection_before_after.png")
plt.show()
plt.close(fig)


## 7. 선택 Feature 표준화와 Gaussian Mixture Model 3군집

GMM은 Feature별 단위와 범위 차이에 민감하므로 **최종 선택된 2~3개 Feature만** `StandardScaler`로 평균 0, 표준편차 1이 되게 변환한다. 원래 5개 전체나 PCA score를 GMM 입력으로 사용하지 않는다.

`GaussianMixture(n_components=3, random_state=42, covariance_type='full')`를 사용한다. full covariance는 각 군집이 서로 다른 분산과 Feature 간 공분산 구조를 가질 수 있게 한다. `n_init=10`, `reg_covar=1e-6`으로 수치 안정성과 초기값 안정성을 높인다. 최종 단계가 Heat Cost 0/1/2여야 하므로 군집 수는 항상 3이다.

출력은 `cluster_raw`, 각 군집 posterior probability, 그중 최대값인 `cluster_confidence`다. `cluster_raw` 번호 자체에는 열적 의미가 없다.


In [ ]:
X_selected = clean_matrix(edge_snapshot, SELECTED_FEATURES)
if not np.isfinite(X_selected.to_numpy()).all():
    raise ValueError("GMM 입력에 NaN/inf가 남아 있습니다.")
if any(X_selected[feature].nunique() < 2 for feature in SELECTED_FEATURES):
    raise ValueError("GMM 입력에 값이 하나뿐인 Feature가 있습니다.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)
gmm = GaussianMixture(
    n_components=3,
    covariance_type="full",
    random_state=RANDOM_STATE,
    n_init=10,
    reg_covar=1e-6,
)
cluster_raw = gmm.fit_predict(X_scaled)
posterior = gmm.predict_proba(X_scaled)
if len(np.unique(cluster_raw)) != 3:
    raise RuntimeError("GMM hard label에 3개 군집이 모두 나타나지 않았습니다.")

edge_snapshot["cluster_raw"] = cluster_raw.astype(int)
for cluster_id in range(3):
    edge_snapshot[f"cluster_prob_{cluster_id}"] = posterior[:, cluster_id]
edge_snapshot["cluster_confidence"] = posterior.max(axis=1)

cluster_counts = (
    edge_snapshot["cluster_raw"].value_counts().sort_index().rename("edge_count").to_frame()
)
cluster_counts["edge_ratio"] = cluster_counts["edge_count"] / len(edge_snapshot)
cluster_counts.index.name = "cluster_raw"
display(cluster_counts)
print(f"GMM BIC={gmm.bic(X_scaled):,.1f} · AIC={gmm.aic(X_scaled):,.1f}")
display(edge_snapshot[["edge_id", "cluster_raw", "cluster_prob_0", "cluster_prob_1",
                       "cluster_prob_2", "cluster_confidence"]].head())


## 7-1. GMM 군집의 2차원 분포 시각화

선택 Feature가 2개면 두 표준화 Feature를 그대로 x/y축에 사용한다. 3개면 그림을 그리기 위해서만 PCA 2차원 projection을 만든다. **PCA는 이 시각화에만 사용하며 GMM 학습에는 사용하지 않았다.**

점 색은 `cluster_raw`이고 투명한 점은 군집 간 겹침을 보여준다. GMM 번호는 단순 식별자이므로 이 그림만 보고 cool/hot 의미를 붙이지 않는다. 출력은 `outputs/ppt_figures/07_gmm_cluster_distribution.png`다.


In [ ]:
if len(SELECTED_FEATURES) == 2:
    plot_xy = X_scaled[:, :2]
    x_label = f"{FEATURE_LABELS[SELECTED_FEATURES[0]]} (표준화)"
    y_label = f"{FEATURE_LABELS[SELECTED_FEATURES[1]]} (표준화)"
    subtitle = "선택된 두 Feature를 직접 표시 · PCA 미사용"
else:
    visualization_pca = PCA(n_components=2, random_state=RANDOM_STATE)
    plot_xy = visualization_pca.fit_transform(X_scaled)
    explained = visualization_pca.explained_variance_ratio_.sum() * 100
    x_label = "PCA 1 (시각화 전용)"
    y_label = "PCA 2 (시각화 전용)"
    subtitle = f"PCA는 시각화 전용 · GMM 학습 미사용 · 설명분산 {explained:.1f}%"

fig, ax = plt.subplots(figsize=(10, 8))
palette = ["#2A9D8F", "#E9C46A", "#E76F51"]
for cluster_id, color in enumerate(palette):
    mask = cluster_raw == cluster_id
    ax.scatter(plot_xy[mask, 0], plot_xy[mask, 1], s=12, alpha=0.45,
               color=color, label=f"cluster_raw {cluster_id}")
ax.set_xlabel(x_label)
ax.set_ylabel(y_label)
ax.set_title(f"GMM 3군집 Feature 공간\n{subtitle}", fontsize=15, fontweight="bold")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
save_figure(fig, "07_gmm_cluster_distribution.png")
plt.show()
plt.close(fig)


## 8. Silhouette Score로 군집 분리도 검증

GMM은 확률모델이지만 `gmm.predict()`의 hard label을 사용해 `silhouette_score(X_scaled, labels)`를 계산한다. 1에 가까우면 같은 군집 내부는 응집되고 다른 군집과 잘 분리되며, 0 부근이면 군집이 많이 겹치고, 음수는 일부 Edge가 다른 군집에 더 가까울 가능성을 뜻한다.

낮은 점수가 나오더라도 군집 수를 자동 변경하지 않는다. 분석 목적상 상대 Heat Cost 0/1/2의 3단계를 유지하고, 점수는 Feature 공간에서의 분리 품질을 정직하게 보고하는 지표로만 사용한다. 출력은 전체 평균과 군집별 sample silhouette 분포 그림이다.


In [ ]:
silhouette_avg = float(silhouette_score(X_scaled, cluster_raw))
silhouette_each = silhouette_samples(X_scaled, cluster_raw)
print(f"Silhouette Score = {silhouette_avg:.4f}")

fig, ax = plt.subplots(figsize=(10, 7))
y_lower = 10
for cluster_id, color in enumerate(palette):
    values = np.sort(silhouette_each[cluster_raw == cluster_id])
    size = len(values)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, values,
                     facecolor=color, edgecolor=color, alpha=0.75)
    ax.text(-0.05, y_lower + size / 2, str(cluster_id), fontweight="bold")
    y_lower = y_upper + 10
ax.axvline(silhouette_avg, color="#C1121F", linestyle="--", linewidth=2,
           label=f"평균 {silhouette_avg:.3f}")
ax.axvline(0, color="#555555", linewidth=1)
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("군집별 Edge (정렬)")
ax.set_yticks([])
ax.set_title("GMM hard label의 Silhouette 분포\n군집 수 3은 유지하며 분리 품질만 해석",
             fontsize=15, fontweight="bold")
ax.legend()
plt.tight_layout()
save_figure(fig, "08_gmm_silhouette_score.png")
plt.show()
plt.close(fig)


## 9. 군집별 Feature profile 해석과 상대 Heat Cost 0/1/2 매핑

`cluster_raw` 번호에는 열적 순서가 없으므로 선택 Feature의 군집별 평균, 중앙값, 전체 기준 z-score와 평균 confidence를 계산한다. z-score는 `StandardScaler`의 전체 평균·표준편차를 사용해 군집 평균이 전체보다 얼마나 높은지 나타낸다.

각 군집의 `relative_heat_score`는 선택 Feature별 `cluster mean z-score × THERMAL_DIRECTION`의 평균이다. 그늘은 방향 -1, 나머지는 +1이다. 점수가 가장 낮은 군집을 Heat Cost 0, 중간을 1, 가장 높은 군집을 2로 매핑한다. 이 점수는 실제 ℃ 공식이 아니라 군집 사이 상대 열노출 순서를 객관적으로 결정하기 위한 기준이다.

해석 문장은 실제 z-score의 절대값이 큰 Feature부터 자동 생성한다. 원래 cluster 번호와 Heat Cost는 별도 컬럼으로 보존한다.


In [ ]:
grouped_snapshot = edge_snapshot.groupby("cluster_raw")
cluster_mean = grouped_snapshot[SELECTED_FEATURES].mean()
cluster_median = grouped_snapshot[SELECTED_FEATURES].median()
cluster_z = (
    cluster_mean - pd.Series(scaler.mean_, index=SELECTED_FEATURES)
) / pd.Series(scaler.scale_, index=SELECTED_FEATURES).replace(0, 1)

profile_base = pd.DataFrame({
    "edge_count": grouped_snapshot.size(),
    "edge_ratio": grouped_snapshot.size() / len(edge_snapshot),
    "mean_cluster_confidence": grouped_snapshot["cluster_confidence"].mean(),
})
cluster_profile = profile_base.copy()
for feature in SELECTED_FEATURES:
    cluster_profile[f"mean__{feature}"] = cluster_mean[feature]
    cluster_profile[f"median__{feature}"] = cluster_median[feature]
    cluster_profile[f"z__{feature}"] = cluster_z[feature]

direction_series = pd.Series({
    feature: THERMAL_DIRECTION[feature] for feature in SELECTED_FEATURES
})
relative_heat = cluster_z.mul(direction_series, axis=1).mean(axis=1)
ordered_clusters = relative_heat.sort_values().index.tolist()
heat_cost_mapping = {cluster_id: rank for rank, cluster_id in enumerate(ordered_clusters)}


def describe_cluster(cluster_id):
    row = cluster_z.loc[cluster_id]
    dominant = row.abs().sort_values(ascending=False).head(min(2, len(row)))
    parts = []
    for feature in dominant.index:
        z_value = float(row[feature])
        level = "높음" if z_value >= 0 else "낮음"
        signed_heat = z_value * THERMAL_DIRECTION[feature]
        thermal = "hotter" if signed_heat > 0 else "cooler"
        parts.append(
            f"{FEATURE_LABELS[feature]} {level}(z={z_value:+.2f}, {thermal} 방향)"
        )
    return "; ".join(parts)


mapping_table = pd.DataFrame({
    "cluster_raw": sorted(cluster_profile.index),
})
mapping_table["relative_heat_score"] = mapping_table["cluster_raw"].map(relative_heat)
mapping_table["heat_cost"] = mapping_table["cluster_raw"].map(heat_cost_mapping).astype(int)
mapping_table["interpretation"] = mapping_table["cluster_raw"].map(describe_cluster)

edge_snapshot["relative_heat_score"] = edge_snapshot["cluster_raw"].map(relative_heat)
edge_snapshot["heat_cost"] = edge_snapshot["cluster_raw"].map(heat_cost_mapping).astype(int)

cluster_profile.index.name = "cluster_raw"
cluster_profile.reset_index().to_csv(
    TABLE_DIR / "gmm_cluster_profile.csv", index=False, encoding="utf-8-sig"
)
mapping_table[["cluster_raw", "heat_cost", "relative_heat_score"]].to_csv(
    TABLE_DIR / "cluster_heatcost_mapping.csv", index=False, encoding="utf-8-sig"
)
mapping_table.to_csv(
    TABLE_DIR / "gmm_cluster_interpretation.csv", index=False, encoding="utf-8-sig"
)

display(cluster_profile.round(3))
display(mapping_table.sort_values("heat_cost").round(3))


## 9-1. 군집별 Edge 비율

각 GMM 군집이 전체 보행 Edge에서 차지하는 비율을 보여준다. 군집이 지나치게 작으면 특정 극단값만 따로 떼었을 가능성을 해석할 때 고려해야 한다. 색은 아직 `cluster_raw` 식별용이며 Heat Cost 색상과 다르다.

출력은 `outputs/ppt_figures/09_cluster_edge_share.png`다.


In [ ]:
share_table = cluster_counts.reset_index()
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(
    share_table["cluster_raw"].astype(str),
    share_table["edge_ratio"] * 100,
    color=palette,
)
for bar, count, ratio in zip(bars, share_table["edge_count"], share_table["edge_ratio"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{count:,}개\n{ratio*100:.1f}%", ha="center", va="bottom")
ax.set_xlabel("cluster_raw")
ax.set_ylabel("전체 Edge 비율 (%)")
ax.set_title("GMM 군집별 Edge 수와 비율", fontsize=15, fontweight="bold")
plt.tight_layout()
save_figure(fig, "09_cluster_edge_share.png")
plt.show()
plt.close(fig)


## 9-2. 군집별 선택 Feature z-score profile

군집 평균을 전체 평균·표준편차 기준 z-score로 바꿔 Feature 단위 차이를 제거한 profile이다. 0은 전체 평균, 양수는 전체보다 높음, 음수는 낮음을 뜻한다. 그늘은 z-score가 높을수록 일반적으로 cooler 방향이라는 점에 유의한다.

출력은 `outputs/ppt_figures/10_cluster_feature_z_profile.png`다.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(SELECTED_FEATURES))
width = 0.23
for offset_index, cluster_id in enumerate(sorted(cluster_z.index)):
    offset = (offset_index - 1) * width
    ax.bar(
        x + offset,
        cluster_z.loc[cluster_id, SELECTED_FEATURES],
        width,
        color=palette[cluster_id],
        label=f"cluster_raw {cluster_id} → Heat Cost {heat_cost_mapping[cluster_id]}",
    )
ax.axhline(0, color="#444444", linewidth=1)
ax.set_xticks(x, [FEATURE_LABELS[name] for name in SELECTED_FEATURES], rotation=20, ha="right")
ax.set_ylabel("군집 평균 z-score")
ax.set_title("GMM 군집별 선택 Feature z-score profile", fontsize=15, fontweight="bold")
ax.legend()
plt.tight_layout()
save_figure(fig, "10_cluster_feature_z_profile.png")
plt.show()
plt.close(fig)


## 9-3. 군집별 Feature profile Heatmap

같은 z-score profile을 Heatmap으로 표현한다. 붉은색은 전체보다 높은 값, 파란색은 낮은 값이다. 이것은 값의 고저를 나타내며 곧바로 hot/cool을 뜻하지 않는다. 예를 들어 `shade_ratio`의 붉은색은 높은 그늘이라 cooler 방향이다. Heat Cost mapping은 별도 열적 방향성 곱으로 계산했다.

출력은 `outputs/ppt_figures/11_cluster_profile_heatmap.png`다.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(
    cluster_z[SELECTED_FEATURES],
    annot=True,
    fmt="+.2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "군집 평균 z-score"},
)
ax.set_xticklabels([FEATURE_LABELS[name] for name in SELECTED_FEATURES], rotation=25, ha="right")
ax.set_yticklabels(
    [f"raw {cluster_id} / cost {heat_cost_mapping[cluster_id]}" for cluster_id in cluster_z.index],
    rotation=0,
)
ax.set_title("군집별 선택 Feature profile Heatmap", fontsize=15, fontweight="bold")
plt.tight_layout()
save_figure(fig, "11_cluster_profile_heatmap.png")
plt.show()
plt.close(fig)


## 10. 전체 Edge에 Heat Cost 결합, 공간 결과 저장, 네트워크 지도

최종 GeoDataFrame에는 `edge_id`, `geometry`, `length_m`, 원래 5개 Feature, `cluster_raw`, posterior probabilities, `cluster_confidence`, 군집 수준 `relative_heat_score`, `heat_cost`가 포함된다. 선택 Feature는 원래 5개 안에 있으므로 동일 이름으로 보존된다.

Heat Cost 0/1/2는 각각 상대적으로 낮음/중간/높은 열노출 군집을 뜻한다. 실제 노면온도 등급이 아니다. 공간 결과는 Parquet와 GeoPackage로 저장하고, 전체망 지도는 3개 색으로 구분한다.


In [ ]:
probability_columns = [f"cluster_prob_{cluster_id}" for cluster_id in range(3)]
final_columns = list(dict.fromkeys([
    "edge_id", "geometry", "length_m", "surface_code", *FEATURES,
    *SELECTED_FEATURES, "cluster_raw", *probability_columns,
    "cluster_confidence", "relative_heat_score", "heat_cost",
]))
edge_result = edge_snapshot[final_columns].copy()
edge_result = gpd.GeoDataFrame(edge_result, geometry="geometry", crs=CRS)

parquet_path = PROCESSED_DIR / "edge_features_0815_1600.parquet"
gpkg_path = PROCESSED_DIR / "edge_cluster_heatcost.gpkg"
edge_result.to_parquet(parquet_path, index=False)
edge_result.to_file(gpkg_path, layer="edge_heatcost", driver="GPKG", mode="w")
print("공간 결과 저장:", parquet_path)
print("공간 결과 저장:", gpkg_path)

heat_colors = {0: "#2A9D8F", 1: "#E9C46A", 2: "#E76F51"}
fig, ax = plt.subplots(figsize=(10, 10))
for heat_cost, color in heat_colors.items():
    part = edge_result[edge_result["heat_cost"] == heat_cost]
    part.plot(ax=ax, color=color, linewidth=1.05, label=f"Heat Cost {heat_cost}")
ax.set_title("송파구 보행망 상대 Heat Cost 0/1/2\n실제 ℃가 아닌 군집 간 상대 순위",
             fontsize=15, fontweight="bold")
ax.legend(loc="lower right")
ax.set_axis_off()
plt.tight_layout()
save_figure(fig, "12_network_heat_cost_map.png")
plt.show()
plt.close(fig)


## 11. 출발·도착 좌표를 보행 Node에 snap하고 연결성 검증

상단 CONFIG의 위경도를 EPSG:5186으로 변환한 뒤 가장 가까운 그래프 Node에 snap한다. 입력점과 Node 사이 거리가 200m를 넘으면 잘못된 위치일 수 있으므로 중단한다. 두 Node가 서로 다른 연결요소면 경로를 억지로 만들지 않고 오류를 낸다.

출력은 snap 거리, snap된 WGS84 좌표, 연결요소 순위다. 기본 좌표는 동봉 보행망의 최대 연결요소 안에서 약 4km의 최단거리 경로가 생기도록 설정했다.


In [ ]:
start_node, start_snap_m = nearest_node(START_LON, START_LAT)
end_node, end_snap_m = nearest_node(END_LON, END_LAT)

if start_node == end_node:
    raise ValueError("출발·도착이 같은 그래프 Node에 snap됐습니다.")
if component_rank[start_node] != component_rank[end_node]:
    raise nx.NetworkXNoPath(
        f"출발 연결요소 {component_rank[start_node]}와 도착 연결요소 "
        f"{component_rank[end_node]}가 다릅니다."
    )
if not nx.has_path(G, start_node, end_node):
    raise nx.NetworkXNoPath("출발·도착 Node 사이에 경로가 없습니다.")

snapped_points = gpd.GeoSeries(
    [Point(start_node), Point(end_node)], crs=CRS
).to_crs(WGS84)
snap_table = pd.DataFrame({
    "point": ["start", "end"],
    "input_lat": [START_LAT, END_LAT],
    "input_lon": [START_LON, END_LON],
    "snapped_lat": [snapped_points.iloc[0].y, snapped_points.iloc[1].y],
    "snapped_lon": [snapped_points.iloc[0].x, snapped_points.iloc[1].x],
    "snap_distance_m": [start_snap_m, end_snap_m],
    "component_rank": [component_rank[start_node], component_rank[end_node]],
})
display(snap_table.round(6))


## 11-1. Heat Cost 경로와 순수 최단거리 경로 계산

각 그래프 구간은 부모 Edge의 Heat Cost를 상속한다. 추천 경로의 가중치는 `length_m × (1 + HEAT_PENALTY × heat_cost)`이고, 기본 `HEAT_PENALTY=1.0`이다. 따라서 Heat Cost 0은 거리 자체, 1은 거리의 2배, 2는 거리의 3배가 된다. 0 Cost Edge도 실제 거리가 남아 있어 불필요한 0비용 순환이나 비정상적 우회를 만들지 않는다.

비교 경로는 `weight=length_m`만 쓰는 순수 최단거리다. 원본 프로젝트의 fast/balanced/cool 산식은 사용하지 않는다. MultiGraph의 평행 구간에서는 해당 weight가 가장 작은 구간을 선택해 실제 route segment를 복원한다.


In [ ]:
heat_by_edge = edge_result.set_index("edge_id")["heat_cost"].to_dict()
for u, v, key, data in G.edges(keys=True, data=True):
    parent_heat = int(heat_by_edge[data["edge_id"]])
    data["heat_cost"] = parent_heat
    data["distance_weight"] = float(data["length_m"])
    data["heat_route_weight"] = float(data["length_m"]) * (
        1 + HEAT_PENALTY * parent_heat
    )


def path_to_segments(graph, node_path, weight_attribute, route_name):
    rows = []
    for order, (u, v) in enumerate(zip(node_path[:-1], node_path[1:]), start=1):
        alternatives = graph.get_edge_data(u, v)
        key, attributes = min(
            alternatives.items(),
            key=lambda item: item[1].get(weight_attribute, np.inf),
        )
        rows.append({
            "route": route_name,
            "segment_order": order,
            "edge_id": attributes["edge_id"],
            "length_m": float(attributes["length_m"]),
            "heat_cost": int(attributes["heat_cost"]),
            "geometry": LineString([u, v]),
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS)


heat_node_path = nx.shortest_path(G, start_node, end_node, weight="heat_route_weight")
shortest_node_path = nx.shortest_path(G, start_node, end_node, weight="distance_weight")
heat_route = path_to_segments(G, heat_node_path, "heat_route_weight", "Heat Cost 기반 추천")
shortest_route = path_to_segments(G, shortest_node_path, "distance_weight", "순수 최단거리")

print(f"Heat Cost 경로 그래프 구간: {len(heat_route):,}")
print(f"최단거리 경로 그래프 구간: {len(shortest_route):,}")


## 12. 두 경로 지표 비교와 자동 해석

각 경로에서 총 거리, 고유 부모 Edge 수, 길이 가중 평균 Heat Cost, Heat Cost 0/1/2의 **거리 구성비**를 계산한다. 길이 가중 평균은 `Σ(length_m × heat_cost) / Σ(length_m)`이므로 짧은 구간이 과도한 영향을 주는 단순 Edge 평균보다 경로 노출을 더 적절히 요약한다.

추천 경로의 추가 거리와 우회율은 순수 최단거리 대비로 계산한다. 자동 해석은 Cost 0 비중이 늘었는지, Cost 2 비중이 줄었는지, 평균 Cost와 거리의 trade-off가 어떠한지를 실제 결과값으로 서술한다.


In [ ]:
def route_metrics(route_gdf):
    total = float(route_gdf["length_m"].sum())
    weighted_mean = float(
        (route_gdf["length_m"] * route_gdf["heat_cost"]).sum() / total
    )
    shares = {
        cost: float(
            100 * route_gdf.loc[route_gdf["heat_cost"] == cost, "length_m"].sum() / total
        )
        for cost in [0, 1, 2]
    }
    return {
        "total_distance_m": total,
        "total_edge_count": int(route_gdf["edge_id"].nunique()),
        "weighted_mean_heat_cost": weighted_mean,
        "heat_cost_0_length_pct": shares[0],
        "heat_cost_1_length_pct": shares[1],
        "heat_cost_2_length_pct": shares[2],
    }


heat_metrics = route_metrics(heat_route)
shortest_metrics = route_metrics(shortest_route)
baseline_distance = shortest_metrics["total_distance_m"]
route_comparison = pd.DataFrame([
    {"route": "Heat Cost 기반 추천", **heat_metrics},
    {"route": "순수 최단거리", **shortest_metrics},
])
route_comparison["extra_distance_vs_shortest_m"] = (
    route_comparison["total_distance_m"] - baseline_distance
)
route_comparison["detour_vs_shortest_pct"] = (
    route_comparison["extra_distance_vs_shortest_m"] / baseline_distance * 100
)
route_comparison.to_csv(
    TABLE_DIR / "route_comparison.csv", index=False, encoding="utf-8-sig"
)
display(route_comparison.round(3))

delta_mean = heat_metrics["weighted_mean_heat_cost"] - shortest_metrics["weighted_mean_heat_cost"]
delta_cost0 = heat_metrics["heat_cost_0_length_pct"] - shortest_metrics["heat_cost_0_length_pct"]
delta_cost2 = heat_metrics["heat_cost_2_length_pct"] - shortest_metrics["heat_cost_2_length_pct"]
detour_pct = float(route_comparison.loc[
    route_comparison["route"] == "Heat Cost 기반 추천", "detour_vs_shortest_pct"
].iloc[0])

route_interpretation = f"""
### 경로 비교 자동 해석

- Heat Cost 기반 추천 경로는 **{heat_metrics['total_distance_m']:.1f}m**, 순수 최단경로는 **{shortest_metrics['total_distance_m']:.1f}m**다.
- 추천 경로의 우회율은 **{detour_pct:.2f}%**다.
- 길이 가중 평균 Heat Cost 차이(추천−최단)는 **{delta_mean:+.3f}**다. 음수이면 추천 경로가 상대적으로 낮은 Cost 구간을 더 이용했다.
- Heat Cost 0 거리 비중 차이는 **{delta_cost0:+.2f}%p**, Heat Cost 2 거리 비중 차이는 **{delta_cost2:+.2f}%p**다.
- 이 비교는 상대적 군집 순위를 이용한 경로 trade-off이며 실제 노면온도 차이(℃)를 뜻하지 않는다.
"""
display(Markdown(route_interpretation))


## 12-A. 최종 추천 경로의 단위 거리당 순수 Heat Cost와 1~100점 경고

이 셀은 **다익스트라가 이미 확정한 `heat_route`를 읽기만 하는 후처리**다. 기존 Edge 가중치, GMM, Heat Cost 0/1/2, 최단경로 및 추천경로를 다시 계산하거나 변경하지 않는다.

매번 GMM을 새로 학습하면 raw cluster 번호가 바뀔 수 있으므로, 현재 실행에서 `heat_cost=2`로 매핑된 raw cluster를 자동으로 찾아 그 posterior probability를 `P(High)`로 사용한다. 대상시각 ASOS 기온은 다음과 같이 0~1로 제한한다.

\[
f(T)=\operatorname{clip}\left(\frac{T-T_{min}}{T_{max}-T_{min}},0,1\right)
\]

각 경로 구간의 순수 열 패널티는 `α × f(T) × P(High)`이고, 추천 경로의 단위 거리당 값과 100점 환산값은 다음과 같다.

\[
U=\frac{\sum_i L_i\,\alpha\,f(T)\,P_i(High)}{\sum_i L_i},\qquad
Score=\operatorname{clip}\left(\frac{U}{\alpha}\times100,0,100\right)
\]

기본 `α=0.5`에서는 `Score = U × 200`이다. 화면용 점수는 반올림 후 1~100으로 제한하고, **40점 이하=쾌적, 41~79점=주의, 80점 이상=산책 자제 경고**로 분류한다. 결과는 그대로 JSON 응답에 넣을 수 있는 `route_safety_payload`와 `get_route_safety_response()`로 제공한다.

> 이 점수는 실측 노면온도나 의학·수의학적 화상 확률이 아니다. 40/80은 서비스 운영용 초기 기준이므로 실제 노면 측정과 사용자 안전 검증을 거쳐 조정해야 한다.


In [ ]:
def temperature_risk_factor(
    temperature_c,
    minimum_c=ROUTE_ALERT_TEMP_MIN_C,
    maximum_c=ROUTE_ALERT_TEMP_MAX_C,
):
    if maximum_c <= minimum_c:
        raise ValueError("ROUTE_ALERT_TEMP_MAX_C는 MIN_C보다 커야 합니다.")
    return float(np.clip(
        (float(temperature_c) - minimum_c) / (maximum_c - minimum_c),
        0.0,
        1.0,
    ))


def round_half_up(value):
    """사용자 화면 점수를 0.5에서 위로 올리는 일반적인 반올림으로 변환한다."""
    return int(math.floor(float(value) + 0.5))


def build_route_safety_payload(route_gdf, edge_gdf, mapping_df, temperature_c):
    if ROUTE_ALERT_ALPHA <= 0:
        raise ValueError("ROUTE_ALERT_ALPHA는 0보다 커야 합니다.")
    if not (1 <= ROUTE_COMFORT_MAX_SCORE < ROUTE_WARNING_MIN_SCORE <= 100):
        raise ValueError("점수 경계는 1 <= 쾌적 상한 < 경고 하한 <= 100이어야 합니다.")
    if route_gdf.empty or float(route_gdf["length_m"].sum()) <= 0:
        raise ValueError("추천 경로가 비어 있거나 총 길이가 0 이하입니다.")

    high_clusters = mapping_df.loc[mapping_df["heat_cost"] == 2, "cluster_raw"]
    if len(high_clusters) != 1:
        raise ValueError(f"heat_cost=2 raw cluster는 정확히 하나여야 합니다: {high_clusters.tolist()}")
    high_cluster_raw = int(high_clusters.iloc[0])
    probability_column = f"cluster_prob_{high_cluster_raw}"
    if probability_column not in edge_gdf.columns:
        raise KeyError(f"고온 군집 posterior 컬럼 누락: {probability_column}")

    probability_by_edge = edge_gdf.set_index("edge_id")[probability_column]
    route_probability = pd.to_numeric(
        route_gdf["edge_id"].map(probability_by_edge), errors="coerce"
    )
    if route_probability.isna().any():
        missing_edge_ids = route_gdf.loc[route_probability.isna(), "edge_id"].unique().tolist()[:10]
        raise ValueError(f"추천 경로 Edge의 P(High) 매핑 실패: {missing_edge_ids}")
    route_probability = route_probability.clip(0.0, 1.0)

    temperature_factor = temperature_risk_factor(temperature_c)
    distance = float(route_gdf["length_m"].sum())
    weighted_mean_p_high = float(
        (route_gdf["length_m"] * route_probability).sum() / distance
    )
    pure_heat_by_segment = ROUTE_ALERT_ALPHA * temperature_factor * route_probability
    pure_heat_numerator = float(
        (route_gdf["length_m"] * pure_heat_by_segment).sum()
    )
    unit_heat_cost = pure_heat_numerator / distance
    raw_score = float(np.clip(unit_heat_cost / ROUTE_ALERT_ALPHA * 100.0, 0.0, 100.0))
    display_score = int(np.clip(round_half_up(raw_score), 1, 100))

    if display_score <= ROUTE_COMFORT_MAX_SCORE:
        status, icon, color = "comfortable", "🟢", "green"
        message = f"오늘 산책길은 아주 쾌적해요! (점수: {display_score}점)"
    elif display_score < ROUTE_WARNING_MIN_SCORE:
        status, icon, color = "caution", "🟡", "yellow"
        message = f"조금 더운 구간이 있으니 물을 챙겨주세요. (점수: {display_score}점)"
    else:
        status, icon, color = "danger", "🚨", "red"
        message = (
            "경고: 오늘 산책 경로는 노면 온도가 너무 높습니다! "
            f"산책을 자제하세요. (점수: {display_score}점)"
        )

    return {
        "route_id": "heat_cost_recommended",
        "target_time_kst": str(actual_target),
        "score": display_score,
        "score_raw_0_100": round(raw_score, 4),
        "unit_heat_cost_0_to_alpha": round(unit_heat_cost, 6),
        "route_distance_m": round(distance, 3),
        "air_temperature_c": round(float(temperature_c), 3),
        "temperature_factor_0_1": round(temperature_factor, 6),
        "weighted_mean_p_high": round(weighted_mean_p_high, 6),
        "high_heat_cluster_raw": high_cluster_raw,
        "alert_alpha": ROUTE_ALERT_ALPHA,
        "status": status,
        "icon": icon,
        "color": color,
        "should_warn": bool(display_score >= ROUTE_WARNING_MIN_SCORE),
        "message": message,
        "thresholds": {
            "comfortable_max": ROUTE_COMFORT_MAX_SCORE,
            "caution_min": ROUTE_COMFORT_MAX_SCORE + 1,
            "caution_max": ROUTE_WARNING_MIN_SCORE - 1,
            "warning_min": ROUTE_WARNING_MIN_SCORE,
        },
        "calibrated_safety_threshold": False,
        "method_note": (
            "경로 확정 후 길이 가중 alpha*f(T)*P(High)를 1~100 UI 점수로 환산; "
            "실측 노면온도 또는 화상 확률이 아님"
        ),
    }


target_temperature_rows = weather_window.loc[
    weather_window["timestamp"] == actual_target, "air_temperature_c"
]
if len(target_temperature_rows) != 1:
    raise ValueError(f"대상시각 기온은 정확히 한 행이어야 합니다: {len(target_temperature_rows)}")
target_air_temperature_c = float(target_temperature_rows.iloc[0])

route_safety_payload = build_route_safety_payload(
    heat_route,
    edge_result,
    mapping_table,
    target_air_temperature_c,
)


def get_route_safety_response():
    """FastAPI/Flask 등의 JSON 응답 body로 바로 반환할 수 있는 payload."""
    return dict(route_safety_payload)


route_safety_path = TABLE_DIR / "route_safety_payload.json"
route_safety_path.write_text(
    json.dumps(route_safety_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
display(Markdown(
    f"### {route_safety_payload['icon']} 사용자 화면 결과\n\n"
    f"**{route_safety_payload['message']}**\n\n"
    f"- 단위 거리당 순수 Heat Cost: `{route_safety_payload['unit_heat_cost_0_to_alpha']}`\n"
    f"- 대상시각 기온 factor: `{route_safety_payload['temperature_factor_0_1']}`\n"
    f"- 추천 경로 길이 가중 P(High): `{route_safety_payload['weighted_mean_p_high']}`\n"
    f"- 백엔드 경고 flag: `{route_safety_payload['should_warn']}`"
))
print(json.dumps(get_route_safety_response(), ensure_ascii=False, indent=2))
print("경고 payload 저장:", route_safety_path)


## 12-1. Heat Cost 기반 추천 경로 지도

전체 Heat Cost 보행망을 흐리게 배경으로 두고 추천 경로를 굵게 표시한다. 출발·도착점도 함께 표시해 경로가 낮은 Cost Edge를 연결하기 위해 어디서 우회하는지 공간적으로 확인한다.

출력은 `outputs/ppt_figures/13_heat_cost_route_map.png`다.


In [ ]:
def add_endpoints(axis):
    points = gpd.GeoSeries([Point(start_node), Point(end_node)], crs=CRS)
    gpd.GeoSeries([points.iloc[0]], crs=CRS).plot(ax=axis, color="#1565C0", marker="o", markersize=90, label="출발")
    gpd.GeoSeries([points.iloc[1]], crs=CRS).plot(ax=axis, color="#C1121F", marker="*", markersize=150, label="도착")


fig, ax = plt.subplots(figsize=(10, 10))
for cost, color in heat_colors.items():
    edge_result[edge_result["heat_cost"] == cost].plot(
        ax=ax, color=color, linewidth=0.45, alpha=0.28
    )
heat_route.plot(ax=ax, color="#6A00F4", linewidth=4, label="Heat Cost 기반 추천")
add_endpoints(ax)
ax.set_title("Heat Cost 기반 추천 산책로", fontsize=15, fontweight="bold")
ax.legend(loc="best")
ax.set_axis_off()
plt.tight_layout()
save_figure(fig, "13_heat_cost_route_map.png")
plt.show()
plt.close(fig)


## 12-2. 순수 최단거리 경로 지도

Heat Cost를 전혀 사용하지 않고 `length_m`만 최소화한 비교 기준 경로다. 동일한 출발·도착점과 지도 범위를 사용하므로 추천 경로와 공간적으로 직접 비교할 수 있다.

출력은 `outputs/ppt_figures/14_shortest_route_map.png`다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
edge_result.plot(ax=ax, color="#D0D0D0", linewidth=0.45, alpha=0.5)
shortest_route.plot(ax=ax, color="#111111", linewidth=4, label="순수 최단거리")
add_endpoints(ax)
ax.set_title("거리만 고려한 순수 최단경로", fontsize=15, fontweight="bold")
ax.legend(loc="best")
ax.set_axis_off()
plt.tight_layout()
save_figure(fig, "14_shortest_route_map.png")
plt.show()
plt.close(fig)


## 12-3. 두 경로 중첩 비교 지도

추천 경로와 최단거리 경로를 같은 지도에 겹쳐 공통 구간과 우회 구간을 확인한다. 선이 완전히 겹치면 거리와 Heat Cost 목적이 같은 경로를 선택했다는 뜻이고, 갈라지는 구간은 열노출 회피의 공간적 trade-off다.

출력은 `outputs/ppt_figures/15_route_overlay_comparison.png`다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
edge_result.plot(ax=ax, color="#E0E0E0", linewidth=0.4, alpha=0.45)
shortest_route.plot(ax=ax, color="#111111", linewidth=5, alpha=0.75, label="순수 최단거리")
heat_route.plot(ax=ax, color="#6A00F4", linewidth=3, alpha=0.9, label="Heat Cost 기반 추천")
add_endpoints(ax)
ax.set_title("동일 출발·도착의 두 경로 중첩 비교", fontsize=15, fontweight="bold")
ax.legend(loc="best")
ax.set_axis_off()
plt.tight_layout()
save_figure(fig, "15_route_overlay_comparison.png")
plt.show()
plt.close(fig)


## 12-4. 거리와 길이 가중 평균 Heat Cost 비교

첫 번째 막대그래프는 추천 경로가 최단거리 대비 얼마나 길어졌는지, 두 번째는 그 대가로 길이 가중 평균 Heat Cost가 얼마나 달라졌는지를 보여준다. Heat Cost는 0~2의 상대 순위이므로 y축을 실제 온도처럼 해석하지 않는다.

출력은 각각 `16_route_distance_comparison.png`, `17_route_mean_heat_comparison.png`다.


In [ ]:
route_colors = ["#6A00F4", "#333333"]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(route_comparison["route"], route_comparison["total_distance_m"], color=route_colors)
for bar, value in zip(bars, route_comparison["total_distance_m"]):
    ax.text(bar.get_x()+bar.get_width()/2, value, f"{value:,.1f}m", ha="center", va="bottom")
ax.set_ylabel("총 거리 (m)")
ax.set_title("두 경로의 총 거리 비교", fontsize=15, fontweight="bold")
plt.tight_layout()
save_figure(fig, "16_route_distance_comparison.png")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(route_comparison["route"], route_comparison["weighted_mean_heat_cost"], color=route_colors)
for bar, value in zip(bars, route_comparison["weighted_mean_heat_cost"]):
    ax.text(bar.get_x()+bar.get_width()/2, value, f"{value:.3f}", ha="center", va="bottom")
ax.set_ylabel("길이 가중 평균 Heat Cost (0~2 상대순위)")
ax.set_title("두 경로의 길이 가중 평균 Heat Cost", fontsize=15, fontweight="bold")
plt.tight_layout()
save_figure(fig, "17_route_mean_heat_comparison.png")
plt.show()
plt.close(fig)


## 12-5. 두 경로의 Heat Cost 0/1/2 거리 구성비

각 경로 전체 길이를 100%로 두고 Heat Cost 0/1/2 구간이 차지하는 거리 비율을 누적 막대로 나타낸다. 추천 경로에서 Cost 0 비중이 늘고 Cost 2가 줄었는지, 또는 네트워크 제약 때문에 변화가 작았는지를 확인한다.

출력은 `outputs/ppt_figures/18_route_heat_cost_composition.png`다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(route_comparison))
for cost in [0, 1, 2]:
    column = f"heat_cost_{cost}_length_pct"
    values = route_comparison[column].to_numpy()
    bars = ax.bar(route_comparison["route"], values, bottom=bottom,
                  color=heat_colors[cost], label=f"Heat Cost {cost}")
    for index, (bar, value) in enumerate(zip(bars, values)):
        if value >= 4:
            ax.text(bar.get_x()+bar.get_width()/2, bottom[index]+value/2,
                    f"{value:.1f}%", ha="center", va="center", fontweight="bold")
    bottom += values
ax.set_ylim(0, 100)
ax.set_ylabel("경로 거리 구성비 (%)")
ax.set_title("두 경로의 Heat Cost 0/1/2 거리 구성", fontsize=15, fontweight="bold")
ax.legend(loc="upper center", ncol=3)
plt.tight_layout()
save_figure(fig, "18_route_heat_cost_composition.png")
plt.show()
plt.close(fig)


## 13. 필수 표·그림·공간 산출물 저장 확인

이 셀은 요청된 9개 CSV, 새 사용자 경고 JSON, 18개 PNG, 2개 공간 파일이 모두 생성됐는지 검사한다. 하나라도 없으면 정상 완료로 오해하지 않도록 오류를 낸다. 표는 PPT 원고와 후속 분석에, 공간 파일은 GIS 검토에 사용할 수 있다.

출력은 파일별 경로와 크기다.


In [ ]:
required_outputs = [
    TABLE_DIR / "feature_summary.csv",
    TABLE_DIR / "feature_correlation.csv",
    TABLE_DIR / "vif_before.csv",
    TABLE_DIR / "vif_after.csv",
    TABLE_DIR / "selected_features.csv",
    TABLE_DIR / "gmm_cluster_profile.csv",
    TABLE_DIR / "gmm_cluster_interpretation.csv",
    TABLE_DIR / "cluster_heatcost_mapping.csv",
    TABLE_DIR / "route_comparison.csv",
    TABLE_DIR / "route_safety_payload.json",
    PROCESSED_DIR / "edge_features_0815_1600.parquet",
    PROCESSED_DIR / "edge_cluster_heatcost.gpkg",
    *[PPT_DIR / name for name in [
        "01_asos_history_0815_1600.png",
        "02_shade_ratio_map_0815_1600.png",
        "03_feature_distributions.png",
        "04_feature_correlation_heatmap.png",
        "05_vif_comparison.png",
        "06_feature_selection_before_after.png",
        "07_gmm_cluster_distribution.png",
        "08_gmm_silhouette_score.png",
        "09_cluster_edge_share.png",
        "10_cluster_feature_z_profile.png",
        "11_cluster_profile_heatmap.png",
        "12_network_heat_cost_map.png",
        "13_heat_cost_route_map.png",
        "14_shortest_route_map.png",
        "15_route_overlay_comparison.png",
        "16_route_distance_comparison.png",
        "17_route_mean_heat_comparison.png",
        "18_route_heat_cost_composition.png",
    ]],
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError("필수 산출물 누락:\n- " + "\n- ".join(missing_outputs))

output_manifest = pd.DataFrame([
    {"path": str(path), "size_kb": path.stat().st_size / 1024}
    for path in required_outputs
])
display(output_manifest.round({"size_kb": 1}))
print(f"필수 산출물 {len(required_outputs)}개 확인 완료")


## 14. 실제 계산 결과 종합 해석

마지막 셀은 앞서 계산한 상관관계, 변수 제거 로그, GMM 품질·군집 비율·Heat Cost mapping, 경로 지표를 실제 숫자로 모아 연구 결과 Markdown을 자동 작성한다. Notebook 설정을 바꿔 재실행하면 서술도 같은 결과에 맞춰 갱신된다.

표현은 일관되게 “상대적으로 열노출이 낮거나 높을 것으로 예상”한다고 제한한다. 실제 노면온도 측정값이나 모델 정확도로 과장하지 않는다.


In [ ]:
top_pair = correlation_pairs.iloc[0]
high_pair_text = (
    "; ".join(
        f"{row.feature_1}–{row.feature_2} (r={row.correlation:+.3f})"
        for row in correlation_pairs[correlation_pairs["above_threshold"]].itertuples()
    ) or f"기준 이상 pair 없음; 최대는 {top_pair.feature_1}–{top_pair.feature_2} (r={top_pair.correlation:+.3f})"
)
removed_text = (
    "; ".join(f"{row.removed_feature} ({row.reason})" for row in selection_log.itertuples())
    if len(selection_log) else "제거 없음"
)

if silhouette_avg >= 0.5:
    silhouette_text = "Feature 공간에서 비교적 뚜렷하게 분리"
elif silhouette_avg >= 0.25:
    silhouette_text = "Feature 공간에서 중간 정도로 분리되지만 일부 겹침 존재"
elif silhouette_avg >= 0:
    silhouette_text = "Feature 공간에서 군집 겹침이 큰 편"
else:
    silhouette_text = "일부 Edge의 hard label 배정이 불안정할 가능성"

cluster_lines = []
for row in mapping_table.sort_values("heat_cost").itertuples():
    ratio = cluster_counts.loc[row.cluster_raw, "edge_ratio"] * 100
    confidence = profile_base.loc[row.cluster_raw, "mean_cluster_confidence"]
    cluster_lines.append(
        f"- raw cluster {row.cluster_raw} → **Heat Cost {row.heat_cost}**: "
        f"Edge {ratio:.1f}%, 평균 confidence {confidence:.3f}; {row.interpretation}. "
        f"relative heat score={row.relative_heat_score:+.3f}"
    )

final_report = f"""
# 분석 결과 요약

## Feature 분석

- 가장 큰 절대 Pearson 상관: **{top_pair.feature_1}–{top_pair.feature_2}, r={top_pair.correlation:+.3f}**
- `|r| ≥ {CORR_THRESHOLD}` pair: {high_pair_text}
- 제거 과정: {removed_text}
- 최종 GMM 입력 Feature: **{', '.join(SELECTED_FEATURES)}**
- 변수 선택에는 PCA를 사용하지 않았고, PCA가 실행된 경우에도 3개 Feature 군집 그림의 2차원 시각화에만 사용했다.

## GMM 분석

- GMM은 선택 Feature를 표준화한 뒤 3개 full-covariance Gaussian component로 적합했다.
- Silhouette Score: **{silhouette_avg:.4f}** — {silhouette_text}. 이 값은 노면온도 모델 정확도가 아니라 내부 군집 분리도다.
{chr(10).join(cluster_lines)}

## Heat Cost 해석

- Heat Cost 0은 선택 Feature의 열적 방향 보정 z-score 평균이 가장 낮은 군집, Heat Cost 2는 가장 높은 군집이다.
- 따라서 Heat Cost 0 Edge는 이 Feature 구성에서 **상대적으로 노면 열노출이 낮을 것으로 예상**되고, Heat Cost 2 Edge는 상대적으로 높을 가능성이 있는 군집이다.
- Heat Cost는 실제 ℃가 아니라 군집 간 상대 순위다.

## 경로 비교

- Heat Cost 기반 추천: **{heat_metrics['total_distance_m']:.1f}m**, 길이 가중 평균 Heat Cost **{heat_metrics['weighted_mean_heat_cost']:.3f}**
- 순수 최단거리: **{shortest_metrics['total_distance_m']:.1f}m**, 길이 가중 평균 Heat Cost **{shortest_metrics['weighted_mean_heat_cost']:.3f}**
- 추천 경로 우회율: **{detour_pct:.2f}%**
- 평균 Heat Cost 차이(추천−최단): **{delta_mean:+.3f}**
- Heat Cost 0 거리 비중 차이: **{delta_cost0:+.2f}%p**; Heat Cost 2 거리 비중 차이: **{delta_cost2:+.2f}%p**
- 이 결과는 거리와 상대적 열노출 회피 사이 trade-off다. 현장 노면온도, 공사·통행 제한, 보행 안전 상태를 직접 검증한 결과는 아니다.

## 사용자 화면 경고

- 최종 추천 경로 Heat 점수: **{route_safety_payload['score']}점**
- 상태: **{route_safety_payload['status']}** · 경고 flag: **{route_safety_payload['should_warn']}**
- 사용자 문구: **{route_safety_payload['message']}**
- 이 40/80 경계는 서비스 운영용 초기값이며 실측 노면온도나 의학·수의학적 위험 확률로 보정된 기준은 아니다.
"""
display(Markdown(final_report))


---

### 재현성과 해석상 주의사항

- 같은 Raw Data, ASOS 응답, 설정, `random_state=42`에서는 동일한 파이프라인을 재현할 수 있다.
- ASOS 시간이 하나라도 빠지면 분석을 중단한다. 서비스키는 Notebook에 저장하지 않는다.
- `surface_code='unknown'`은 원본과 같은 기본 흡수율 0.75를 사용하며, 포장재 매칭률과 함께 해석해야 한다.
- GMM은 비지도 군집 분석이다. posterior probability와 Silhouette Score는 내부 구조를 설명하지만 실제 노면온도 관측을 대체하지 않는다.
- 추천 경로의 1~100점과 40/80 경계는 UI용 후처리다. 현장 노면 측정과 안전성 검증 전에는 임상·수의학적 판정으로 사용하지 않는다.
- 경로 결과는 동봉 보행망의 연결성과 Edge 속성에 의존한다. 실제 이용 전 최신 통행 가능 여부와 보행 안전을 별도로 확인해야 한다.
